# Pumping Stations Positioning — Cyprus WWTP2 Networks

This version uses the corrected WWTP2 coordinates, applies stricter cost-control criteria to budgeted scenarios, limits the number of new pumping stations proportionally to the number of existing pumping stations, and adds a final cost summary using 1.5 M€ per new pumping station.

## BLOCK A — Setup & Imports

In [1]:
import sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")

REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "geopandas": "geopandas",
    "fiona": "fiona", "shapely": "shapely", "networkx": "networkx",
    "rasterio": "rasterio", "scipy": "scipy", "pyproj": "pyproj",
}

missing = [pkg for mod, pkg in REQUIRED.items() if not importlib.util.find_spec(mod)]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *missing])

from pathlib import Path
import os, math, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import networkx as nx
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import Point, LineString
from scipy.spatial import cKDTree
from pyproj import Transformer

print("Environment ready.")


Environment ready.


## BLOCK B — Paths & Scenario Configuration

In [2]:
PROJECT_DIR  = Path(r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project")
AREAS_DIR    = PROJECT_DIR / "areas"
AREAS2_DIR   = PROJECT_DIR / "areas2"
MANUAL_DIR   = PROJECT_DIR / "Results" / "After_Manual_Networks_Correction"
OUTPUT_DIR   = PROJECT_DIR / "Results" / "PunpingStations_Positioning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_TIF = AREAS_DIR / "elevation1.tif"

# WWTP2 coordinates (WGS84) — FIXED / USER-CONFIRMED
# Important: shapely/pyproj use always_xy=True, therefore the correct order is:
#   longitude, latitude = x, y
WWTP2_LONLAT = (33.611278405275385, 34.978016504185796)
WWTP2_LATLON = (34.978016504185796, 33.611278405275385)
WWTP2_LON = WWTP2_LONLAT[0]
WWTP2_LAT = WWTP2_LONLAT[1]

CRS_PROJECTED = "EPSG:32636"

# Pipe cost used consistently with the previous network notebooks / summaries.
PIPE_COST_EUR_PER_M = 400.0

# Additional pumping-station CAPEX assumption for this screening-level analysis.
NEW_PUMPING_STATION_CAPEX_EUR = 1_500_000.0

# ── Scenario definitions ────────────────────────────────────
# algorithm_family drives parameter differentiation (see BLOCK C)
SCENARIOS = [
    {
        "key":              "dijkstra",
        "label":            "Rooted Dijkstra",
        "algorithm_family": "non_budgeted",
        "gpkg": MANUAL_DIR / "dijkstra_final_manual_correction"           / "final_manual_corrected_network.gpkg",
    },
    {
        "key":              "prim_steiner",
        "label":            "Prim-Steiner",
        "algorithm_family": "non_budgeted",
        "gpkg": MANUAL_DIR / "prim_steiner_final_manual_correction"       / "final_manual_corrected_network.gpkg",
    },
    {
        "key":              "shared_trunk",
        "label":            "Shared-Trunk",
        "algorithm_family": "budgeted",
        "gpkg": MANUAL_DIR / "shared_trunk_final_manual_correction"       / "final_manual_corrected_network.gpkg",
    },
    {
        "key":              "urban_priority",
        "label":            "Urban Priority",
        "algorithm_family": "budgeted",
        "gpkg": MANUAL_DIR / "urban_priority_final_manual_correction"     / "final_manual_corrected_network.gpkg",
    },
    {
        "key":              "municipality_targets",
        "label":            "Municipality Targets",
        "algorithm_family": "budgeted",
        "gpkg": MANUAL_DIR / "municipality_targets_final_manual_correction" / "final_manual_corrected_network.gpkg",
    },
]

print("Output directory:", OUTPUT_DIR)
print("DEM found:", DEM_TIF.exists())
print("Scenarios configured:", [s["key"] for s in SCENARIOS])
print("WWTP2_LONLAT:", WWTP2_LONLAT)
print("WWTP2_LATLON:", WWTP2_LATLON)
print(f"Pipe cost assumption: {PIPE_COST_EUR_PER_M:,.0f} €/m")
print(f"New pumping station CAPEX assumption: {NEW_PUMPING_STATION_CAPEX_EUR/1e6:,.2f} M€/station")

Output directory: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning
DEM found: True
Scenarios configured: ['dijkstra', 'prim_steiner', 'shared_trunk', 'urban_priority', 'municipality_targets']
WWTP2_LONLAT: (33.611278405275385, 34.978016504185796)
WWTP2_LATLON: (34.978016504185796, 33.611278405275385)
Pipe cost assumption: 400 €/m
New pumping station CAPEX assumption: 1.50 M€/station


## BLOCK C — Hydraulic Design and Necessity-Based Pumping-Station Selection

This updated version does **not** use the proportional cap as a hard limit on all new pumping stations. The cap is applied only to **optional** candidates. Candidates classified as **mandatory** are always retained because they represent technically critical gravity limitations or conditions similar to the context of existing pumping stations.

In [3]:
# ── Common design parameters (same pipe standard for all scenarios) ──
DESIGN_DIAMETER_MM          = 300.0
DESIGN_VELOCITY_M_S         = 0.60
DESIGN_MANNING_N            = 0.013
MIN_COVER_M                 = 1.20
DEM_SMOOTHING_RADIUS_M      = 75.0
SNAP_DISTANCE_EXISTING_PS_M = 500.0   # max dist to classify existing PS as WWTP2/WWTP1

# Manning minimum gravity slope
def manning_min_slope(diameter_mm, v_min=0.60, n=0.013):
    D = diameter_mm / 1000.0
    R = D / 4.0
    if D <= 0 or R <= 0:
        return np.nan
    return ((v_min * n) / (R ** (2.0 / 3.0))) ** 2.0

S_MIN = manning_min_slope(DESIGN_DIAMETER_MM, DESIGN_VELOCITY_M_S, DESIGN_MANNING_N)
print(f"Manning minimum slope  D={DESIGN_DIAMETER_MM:.0f}mm: {S_MIN:.5f}  ({100*S_MIN:.3f} %)")

# ── Necessity-based selection parameters ─────────────────────
# The previous cost-optimised version used the proportional cap as a hard final limit:
#   non_budgeted = ceil(existing_PS_count * 0.75)
#   budgeted     = ceil(existing_PS_count * 0.35)
#
# This updated version uses the same cap ONLY for OPTIONAL stations.
# MANDATORY stations are never removed by the cap.
#
# Candidate classes:
#   1. mandatory
#      Kept regardless of the cap. These are severe gravity-depth exceedances,
#      adverse terrain situations, or candidates with a terrain/network signature
#      similar to existing pumping-station contexts.
#
#   2. optional
#      Technically useful but less critical. These are ranked by priority score
#      and then capped to control costs.
#
#   3. screened_out
#      Too weak, too local, or insufficiently justified.
#
# Existing-station calibration:
#   Existing pumping stations are used as calibration evidence, not only as a number.
#   The model computes local terrain/network statistics around existing PS and marks
#   new candidates with similar or more severe context as "existing-like critical".

EXISTING_PS_CALIBRATION_RADIUS_M = 750.0
EXISTING_LIKE_RELIEF_QUANTILE    = 0.50
EXISTING_LIKE_SCORE_QUANTILE     = 0.50

# Detection and mandatory rules are HYDRAULIC and therefore identical across
# scenario families. Only the optional-station cap differs, as a cost lever:
# the budgeted family caps optional stations more tightly for cost control.
_HYDRAULIC_COMMON = {
    # Raw / spatial filtering (candidate detection) -- hydraulic, shared
    "min_station_spacing_m":          1_000.0,
    "min_upstream_pipe_length_m":     1_200.0,
    "min_depth_excess_final_m":       2.75,
    "high_depth_always_keep_m":       5.00,
    "min_depth_excess_raw_m":         1.75,
    "min_path_dist_between_raw_m":    800.0,

    # Mandatory / essential rules -- hydraulic necessity, shared
    "mandatory_depth_excess_m":                  7.00,
    "mandatory_upstream_pipe_length_m":          2_500.0,
    "mandatory_adverse_rise_m":                  3.00,
    "mandatory_existing_like_min_depth_excess_m": 3.50,
    "existing_like_min_upstream_pipe_length_m":   1_500.0,
}

FAMILY_PARAMS = {
    "non_budgeted": {
        **_HYDRAULIC_COMMON,
        # Optional cap -- cost lever (relaxed: full-coverage reference)
        "max_new_ps_ratio_to_existing":   0.75,
        "hard_max_new_ps":                30,
        "description": "Non-budgeted: full-coverage reference; mandatory stations kept, optional stations lightly capped",
    },
    "budgeted": {
        **_HYDRAULIC_COMMON,
        # Optional cap -- cost lever (tightened for cost control)
        "max_new_ps_ratio_to_existing":   0.35,
        "hard_max_new_ps":                15,
        "description": "Budgeted: same hydraulic screening; optional stations capped more tightly for cost control",
    },
}

for fam, p in FAMILY_PARAMS.items():
    print(f"\n  [{fam}]  {p['description']}")
    for k, v in p.items():
        if k != "description":
            print(f"    {k}: {v}")

Manning minimum slope  D=300mm: 0.00192  (0.192 %)

  [non_budgeted]  Non-budgeted: full-coverage reference; mandatory stations kept, optional stations lightly capped
    min_station_spacing_m: 1000.0
    min_upstream_pipe_length_m: 1200.0
    min_depth_excess_final_m: 2.75
    high_depth_always_keep_m: 5.0
    min_depth_excess_raw_m: 1.75
    min_path_dist_between_raw_m: 800.0
    mandatory_depth_excess_m: 7.0
    mandatory_upstream_pipe_length_m: 2500.0
    mandatory_adverse_rise_m: 3.0
    mandatory_existing_like_min_depth_excess_m: 3.5
    existing_like_min_upstream_pipe_length_m: 1500.0
    max_new_ps_ratio_to_existing: 0.75
    hard_max_new_ps: 30

  [budgeted]  Budgeted: same hydraulic screening; optional stations capped more tightly for cost control
    min_station_spacing_m: 1000.0
    min_upstream_pipe_length_m: 1200.0
    min_depth_excess_final_m: 2.75
    high_depth_always_keep_m: 5.0
    min_depth_excess_raw_m: 1.75
    min_path_dist_between_raw_m: 800.0
    mandatory_dept

## BLOCK D — Helper Functions

In [4]:
# ── GPKG utilities ──────────────────────────────────────────
def list_layers(path):
    return list(fiona.listlayers(str(path)))

def read_layer_auto(path, keywords, geometry_types=None):
    """Pick best matching layer by keyword score."""
    layers = list_layers(path)
    scored = []
    for lyr in layers:
        low = lyr.lower()
        score = sum(kw.lower() in low for kw in keywords)
        if score == 0:
            continue
        sample = gpd.read_file(path, layer=lyr, rows=5)
        if geometry_types:
            gtypes = set(sample.geom_type.dropna().unique())
            if not gtypes.intersection(geometry_types):
                continue
        scored.append((score, lyr))
    if not scored:
        return None, None
    scored.sort(reverse=True)
    best = scored[0][1]
    return gpd.read_file(path, layer=best), best

def iter_lines(geom):
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "LineString":
        return [geom]
    if geom.geom_type == "MultiLineString":
        return list(geom.geoms)
    return []

def round_node(xy, prec=3):
    return (round(float(xy[0]), prec), round(float(xy[1]), prec))

# ── DEM sampling ────────────────────────────────────────────
def sample_dem(points_gdf, dem_tif, target_crs, output_col="ground_elev_m"):
    """Sample raster DEM at point geometries. Returns 0 if DEM unavailable."""
    if not Path(str(dem_tif)).exists():
        points_gdf = points_gdf.copy()
        points_gdf[output_col] = 0.0
        print("    [WARNING] DEM not found — using elevation = 0 m (flat terrain fallback)")
        return points_gdf

    with rasterio.open(str(dem_tif)) as src:
        dem_crs    = src.crs
        dem_nodata = src.nodata
        pts_proj   = points_gdf.to_crs(dem_crs)
        elevs = []
        for geom in pts_proj.geometry:
            try:
                row, col = src.index(geom.x, geom.y)
                val = float(src.read(1, window=rasterio.windows.Window(col, row, 1, 1))[0, 0])
                if dem_nodata is not None and val == dem_nodata:
                    val = np.nan
            except Exception:
                val = np.nan
            elevs.append(val)

    points_gdf = points_gdf.copy()
    points_gdf[output_col] = elevs
    missing_pct = 100 * sum(np.isnan(v) for v in elevs) / max(len(elevs), 1)
    print(f"    DEM sampled {len(elevs)} points — missing: {missing_pct:.1f}%")
    return points_gdf

def smooth_elevations(nodes_gdf, radius_m=75.0,
                      col_in="ground_elev_m", col_out="ground_elev_used_m"):
    coords = np.array([(g.x, g.y) for g in nodes_gdf.geometry])
    elev   = pd.to_numeric(nodes_gdf[col_in], errors="coerce").to_numpy()
    tree   = cKDTree(coords)
    nbrs   = tree.query_ball_point(coords, r=radius_m)
    smooth = []
    for ids in nbrs:
        vals = elev[ids]
        vals = vals[~np.isnan(vals)]
        smooth.append(float(np.median(vals)) if len(vals) > 0 else np.nan)
    nodes_gdf = nodes_gdf.copy()
    nodes_gdf[col_out] = smooth
    return nodes_gdf

# ── Sanitize & write ────────────────────────────────────────
def sanitize(gdf):
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if pd.api.types.is_object_dtype(out[col]):
            out[col] = out[col].astype(str)
    return out

def write_layer(gdf, gpkg_path, layer_name):
    if gdf is None or len(gdf) == 0:
        print(f"    (skip empty) {layer_name}")
        return
    sanitize(gdf).to_file(str(gpkg_path), layer=layer_name, driver="GPKG")
    print(f"    ✓ {layer_name}  ({len(gdf)} features)")

print("Helper functions defined.")


Helper functions defined.


## BLOCK E — WWTP2 Point & Existing Pumping Stations

In [5]:
# ── WWTP2 projected point ───────────────────────────────────
# Transformer uses always_xy=True, so input MUST be longitude, latitude.
if not (30.0 <= WWTP2_LON <= 36.0 and 32.0 <= WWTP2_LAT <= 36.0):
    raise ValueError(
        "WWTP2 coordinates look invalid for Cyprus. "
        "Check that WWTP2_LONLAT is (longitude, latitude), not (latitude, longitude)."
    )

t = Transformer.from_crs("EPSG:4326", CRS_PROJECTED, always_xy=True)
wx, wy = t.transform(WWTP2_LON, WWTP2_LAT)
WWTP2_POINT = Point(wx, wy)

print("WWTP2 coordinate mode: MANUAL_FIXED_WGS84")
print(f"WWTP2_LONLAT: ({WWTP2_LON:.12f}, {WWTP2_LAT:.12f})")
print(f"WWTP2_LATLON: ({WWTP2_LAT:.12f}, {WWTP2_LON:.12f})")
print(f"WWTP2 projected: ({wx:.3f}, {wy:.3f})  [{CRS_PROJECTED}]")

# ── Load existing pumping stations ──────────────────────────
def load_existing_ps():
    search_paths = [
        AREAS2_DIR / "storages.gpkg",
        AREAS_DIR  / "storages.gpkg",
        AREAS2_DIR / "pumps.gpkg",
        AREAS_DIR  / "pumps.gpkg",
        AREAS2_DIR / "pump_stations.gpkg",
        AREAS_DIR  / "pump_stations.gpkg",
    ]
    for p in search_paths:
        if p.exists():
            layers = list_layers(p)
            if layers:
                gdf = gpd.read_file(p, layer=layers[0]).to_crs(CRS_PROJECTED)
                print(f"Existing PS loaded: {p.name}  ({len(gdf)} features)")
                gdf["ps_source_file"] = p.name
                return gdf
    print("[WARNING] No existing pumping station file found.")
    return None

existing_ps = load_existing_ps()
EXISTING_PS_COUNT_REFERENCE = int(len(existing_ps)) if existing_ps is not None else 0
print(f"Existing PS count used for optional proportional cap: {EXISTING_PS_COUNT_REFERENCE}")

# ── Calibrate max gravity cover from existing PS depths ─────
MAX_GRAVITY_COVER_M = 8.0   # default fallback
if existing_ps is not None and "MaxDepth" in existing_ps.columns:
    depths = pd.to_numeric(existing_ps["MaxDepth"], errors="coerce").dropna()
    if len(depths) > 0:
        p90 = float(depths.quantile(0.90))
        MAX_GRAVITY_COVER_M = max(8.0, p90)
        print(f"\nMaxDepth P90 from existing PS: {p90:.2f} m")

print(f"MAX_GRAVITY_COVER_M used for all scenarios: {MAX_GRAVITY_COVER_M:.2f} m")


def scenario_new_ps_cap(family_params, existing_ps_count):
    """Return the optional-station cap. Mandatory stations are not capped."""
    if existing_ps_count is None or existing_ps_count <= 0:
        return int(family_params.get("hard_max_new_ps", 999999))
    ratio_cap = int(math.ceil(existing_ps_count * float(family_params.get("max_new_ps_ratio_to_existing", 1.0))))
    hard_cap = int(family_params.get("hard_max_new_ps", ratio_cap))
    return max(1, min(ratio_cap, hard_cap))

for fam, p in FAMILY_PARAMS.items():
    print(f"Optional NEW PS cap for {fam}: {scenario_new_ps_cap(p, EXISTING_PS_COUNT_REFERENCE)}")

WWTP2 coordinate mode: MANUAL_FIXED_WGS84
WWTP2_LONLAT: (33.611278405275, 34.978016504186)
WWTP2_LATLON: (34.978016504186, 33.611278405275)
WWTP2 projected: (555795.446, 3870775.810)  [EPSG:32636]
Existing PS loaded: storages.gpkg  (22 features)
Existing PS count used for optional proportional cap: 22

MaxDepth P90 from existing PS: 8.63 m
MAX_GRAVITY_COVER_M used for all scenarios: 8.63 m
Optional NEW PS cap for non_budgeted: 17
Optional NEW PS cap for budgeted: 8


## BLOCK F — Core Hydraulic Detection Function

The updated function keeps the original hydraulic detection logic, but changes the final selection rule:

1. raw gravity-depth candidates are detected;
2. nearby duplicates are removed;
3. candidates are classified as mandatory, optional, or screened out;
4. mandatory candidates are always retained;
5. the proportional cap is applied only to optional candidates.

In [6]:
def build_existing_ps_signature(existing_ps_gdf, pipe_gdfs, dem_tif, target_crs,
                                calibration_radius_m=EXISTING_PS_CALIBRATION_RADIUS_M):
    """
    Build a simple calibration signature from the context of existing pumping stations.

    The signature is not interpreted as proof of the exact engineering reason for each
    historical station. It is used as calibration evidence: if an existing station is
    located in a certain terrain/network context, then new candidate stations with a
    similar or more severe context should be considered technically plausible and may
    become mandatory.

    Parameters
    ----------
    existing_ps_gdf : GeoDataFrame
        Existing pumping stations.
    pipe_gdfs : list[GeoDataFrame]
        Existing pipe layers used to describe the surrounding network context.
    dem_tif : Path
        DEM raster.
    target_crs : str
        Projected CRS used by the network.
    calibration_radius_m : float
        Search radius around each existing pumping station.

    Returns
    -------
    dict
        Quantile-based local terrain/network thresholds.
    """
    default = {
        "valid": False,
        "calibration_radius_m": calibration_radius_m,
        "n_existing_ps_used": 0,
        "local_relief_p50_m": np.nan,
        "local_relief_p75_m": np.nan,
        "local_terrain_score_p50": np.nan,
        "local_terrain_score_p75": np.nan,
        "nearest_pipe_distance_p75_m": np.nan,
    }

    if existing_ps_gdf is None or len(existing_ps_gdf) == 0:
        print("    [Calibration] No existing PS available. Existing-like mandatory rule disabled.")
        return default

    psg = existing_ps_gdf.copy()
    if psg.crs is None:
        psg = psg.set_crs(target_crs)
    else:
        psg = psg.to_crs(target_crs)

    psg = psg[psg.geometry.notna() & (~psg.geometry.is_empty)].copy()
    if len(psg) == 0:
        print("    [Calibration] Existing PS layer is empty after geometry cleaning.")
        return default

    # Force point geometries. If the input has polygons/lines, use centroids.
    psg["geometry"] = psg.geometry.apply(lambda g: g if g.geom_type == "Point" else g.centroid)
    psg = sample_dem(psg, dem_tif, target_crs, output_col="existing_ps_ground_elev_m")

    # Collect pipe vertices around the existing network.
    pipe_points = []
    for pipes in pipe_gdfs:
        if pipes is None or len(pipes) == 0:
            continue
        pp = pipes.copy()
        if pp.crs is None:
            pp = pp.set_crs(target_crs)
        else:
            pp = pp.to_crs(target_crs)
        for geom in pp.geometry:
            for line in iter_lines(geom):
                coords = list(line.coords)
                # Use vertices as a lightweight terrain sample along the existing pipe network.
                for x, y in coords:
                    pipe_points.append((round(float(x), 2), round(float(y), 2)))

    pipe_points = list(dict.fromkeys(pipe_points))
    if len(pipe_points) == 0:
        print("    [Calibration] No pipe vertices found around existing PS. Existing-like mandatory rule disabled.")
        return default

    # Limit very large vertex sets for speed while preserving spatial spread.
    max_pipe_sample_points = 40_000
    if len(pipe_points) > max_pipe_sample_points:
        idx = np.linspace(0, len(pipe_points) - 1, max_pipe_sample_points).astype(int)
        pipe_points = [pipe_points[i] for i in idx]

    pipe_pts_gdf = gpd.GeoDataFrame(
        {"pipe_pt_id": range(len(pipe_points))},
        geometry=[Point(xy) for xy in pipe_points],
        crs=target_crs,
    )
    pipe_pts_gdf = sample_dem(pipe_pts_gdf, dem_tif, target_crs, output_col="pipe_ground_elev_m")

    coords = np.array([(g.x, g.y) for g in pipe_pts_gdf.geometry])
    elevs = pd.to_numeric(pipe_pts_gdf["pipe_ground_elev_m"], errors="coerce").to_numpy()

    if len(coords) == 0:
        return default

    tree = cKDTree(coords)

    records = []
    for _, row in psg.iterrows():
        pt = row.geometry
        idxs = tree.query_ball_point([pt.x, pt.y], r=calibration_radius_m)
        nearest_dist, _ = tree.query([pt.x, pt.y], k=1)

        vals = elevs[idxs] if len(idxs) else np.array([])
        vals = vals[~np.isnan(vals)]

        if len(vals) >= 2:
            local_relief = float(np.nanmax(vals) - np.nanmin(vals))
            local_std = float(np.nanstd(vals))
        else:
            local_relief = 0.0
            local_std = 0.0

        local_count = int(len(idxs))
        terrain_score = local_relief * np.log1p(local_count / 10.0)

        records.append({
            "nearest_pipe_distance_m": float(nearest_dist),
            "local_relief_m": local_relief,
            "local_elevation_std_m": local_std,
            "local_pipe_vertex_count": local_count,
            "local_terrain_score": float(terrain_score),
        })

    sig_df = pd.DataFrame(records)
    sig_df = sig_df.replace([np.inf, -np.inf], np.nan)

    valid_df = sig_df.dropna(subset=["local_relief_m", "local_terrain_score"])
    if len(valid_df) == 0:
        print("    [Calibration] Existing PS signature could not be computed.")
        return default

    signature = {
        "valid": True,
        "calibration_radius_m": calibration_radius_m,
        "n_existing_ps_used": int(len(valid_df)),
        "local_relief_p50_m": float(valid_df["local_relief_m"].quantile(EXISTING_LIKE_RELIEF_QUANTILE)),
        "local_relief_p75_m": float(valid_df["local_relief_m"].quantile(0.75)),
        "local_terrain_score_p50": float(valid_df["local_terrain_score"].quantile(EXISTING_LIKE_SCORE_QUANTILE)),
        "local_terrain_score_p75": float(valid_df["local_terrain_score"].quantile(0.75)),
        "nearest_pipe_distance_p75_m": float(valid_df["nearest_pipe_distance_m"].quantile(0.75)),
    }

    print("    [Calibration] Existing PS context signature:")
    print(f"      existing PS used: {signature['n_existing_ps_used']}")
    print(f"      local relief P50/P75: {signature['local_relief_p50_m']:.2f} / {signature['local_relief_p75_m']:.2f} m")
    print(f"      terrain score P50/P75: {signature['local_terrain_score_p50']:.2f} / {signature['local_terrain_score_p75']:.2f}")
    print(f"      nearest pipe distance P75: {signature['nearest_pipe_distance_p75_m']:.1f} m")

    return signature


def add_candidate_local_terrain_stats(raw_gdf, nodes_gdf,
                                      radius_m=EXISTING_PS_CALIBRATION_RADIUS_M):
    """Add local terrain-relief statistics around each raw candidate."""
    if raw_gdf is None or len(raw_gdf) == 0:
        return raw_gdf

    nodes = nodes_gdf.copy()
    if "ground_elev_used_m" not in nodes.columns:
        nodes["ground_elev_used_m"] = nodes.get("ground_elev_m", np.nan)

    coords = np.array([(g.x, g.y) for g in nodes.geometry])
    elevs = pd.to_numeric(nodes["ground_elev_used_m"], errors="coerce").to_numpy()

    if len(coords) == 0:
        raw_gdf = raw_gdf.copy()
        raw_gdf["local_relief_m"] = np.nan
        raw_gdf["local_elevation_std_m"] = np.nan
        raw_gdf["local_network_node_count"] = 0
        raw_gdf["local_terrain_score"] = np.nan
        return raw_gdf

    tree = cKDTree(coords)

    reliefs, stds, counts, scores = [], [], [], []
    for _, row in raw_gdf.iterrows():
        idxs = tree.query_ball_point([row["station_x"], row["station_y"]], r=radius_m)
        vals = elevs[idxs] if len(idxs) else np.array([])
        vals = vals[~np.isnan(vals)]

        if len(vals) >= 2:
            relief = float(np.nanmax(vals) - np.nanmin(vals))
            std = float(np.nanstd(vals))
        else:
            relief = 0.0
            std = 0.0

        count = int(len(idxs))
        score = relief * np.log1p(count / 10.0)

        reliefs.append(relief)
        stds.append(std)
        counts.append(count)
        scores.append(float(score))

    raw_gdf = raw_gdf.copy()
    raw_gdf["local_relief_m"] = reliefs
    raw_gdf["local_elevation_std_m"] = stds
    raw_gdf["local_network_node_count"] = counts
    raw_gdf["local_terrain_score"] = scores
    return raw_gdf


def mandatory_candidate_reason(row, family_params, existing_ps_signature=None):
    """
    Decide whether a deduplicated candidate is mandatory.

    Mandatory candidates are kept independently of the optional proportional cap.
    """
    p = family_params

    depth = float(row.get("depth_excess_m", 0.0) or 0.0)
    up    = float(row.get("upstream_pipe_length_m", 0.0) or 0.0)
    rise  = float(row.get("adverse_rise_m", 0.0) or 0.0)
    relief = float(row.get("local_relief_m", 0.0) or 0.0)
    terrain_score = float(row.get("local_terrain_score", 0.0) or 0.0)

    reasons = []

    # Very high gravity-depth exceedance: this was previously "always keep".
    if depth >= float(p.get("high_depth_always_keep_m", np.inf)):
        reasons.append(
            f"mandatory_high_depth_excess>= {p['high_depth_always_keep_m']}m"
        )

    # Extreme depth exceedance regardless of other metrics.
    if depth >= float(p.get("mandatory_depth_excess_m", np.inf)):
        reasons.append(
            f"mandatory_extreme_depth_excess>= {p['mandatory_depth_excess_m']}m"
        )

    # Terrain rises against the direction to the outlet and the branch is significant.
    if (
        rise >= float(p.get("mandatory_adverse_rise_m", np.inf))
        and up >= float(p.get("mandatory_upstream_pipe_length_m", np.inf))
    ):
        reasons.append(
            "mandatory_adverse_terrain_rise_with_significant_upstream_network"
        )

    # Existing-like terrain/network context.
    sig = existing_ps_signature or {}
    if sig.get("valid", False):
        relief_thr = float(sig.get("local_relief_p50_m", np.inf))
        relief_thr_severe = float(sig.get("local_relief_p75_m", np.inf))
        score_thr = float(sig.get("local_terrain_score_p50", np.inf))
        score_thr_severe = float(sig.get("local_terrain_score_p75", np.inf))

        existing_like_basic = (
            np.isfinite(relief_thr)
            and np.isfinite(score_thr)
            and relief_thr > 0
            and score_thr > 0
            and relief >= relief_thr
            and terrain_score >= score_thr
            and depth >= float(p.get("mandatory_existing_like_min_depth_excess_m", np.inf))
            and up >= float(p.get("existing_like_min_upstream_pipe_length_m", np.inf))
        )

        existing_like_severe = (
            np.isfinite(relief_thr_severe)
            and np.isfinite(score_thr_severe)
            and relief_thr_severe > 0
            and score_thr_severe > 0
            and relief >= relief_thr_severe
            and terrain_score >= score_thr_severe
            and depth >= float(p.get("min_depth_excess_final_m", np.inf))
            and up >= 0.50 * float(p.get("min_upstream_pipe_length_m", 0.0))
        )

        if existing_like_basic:
            reasons.append("mandatory_existing_PS_like_terrain_network_signature")
        if existing_like_severe:
            reasons.append("mandatory_severe_existing_PS_like_context")

    return bool(reasons), ";".join(reasons) if reasons else "optional_or_screened"


def detect_ps_candidates(pipes_gdf, wwtp2_point, dem_tif, target_crs,
                          family_params, max_gravity_cover_m,
                          max_final_candidates=None,
                          existing_ps_signature=None):
    """
    Detect pumping station candidates on a new pipe network.

    Updated decision logic:
      1. Convert the new WWTP2 pipe layer into a graph.
      2. Orient each connected component toward the WWTP2 point.
      3. Sample DEM elevations at graph nodes.
      4. Propagate a gravity invert profile from terminal branches toward WWTP2.
      5. Create a raw candidate where gravity flow would require excessive cover.
      6. Add local terrain/network statistics around each candidate.
      7. Deduplicate nearby candidates.
      8. Classify candidates as mandatory / optional / screened out.
      9. Keep all mandatory candidates.
     10. Apply the proportional cap only to optional candidates.

    Returns
    -------
    final_ps_gdf, raw_ps_gdf  (both GeoDataFrames in target_crs)
    """
    p = family_params  # shorthand

    # ── 1. Build undirected graph ────────────────────────────
    G = nx.Graph()
    for _, row in pipes_gdf.iterrows():
        for line in iter_lines(row.geometry):
            coords = list(line.coords)
            for a, b in zip(coords[:-1], coords[1:]):
                a_r = round_node(a)
                b_r = round_node(b)
                G.add_edge(a_r, b_r, length_m=Point(a).distance(Point(b)))

    if G.number_of_nodes() == 0:
        print("    [SKIP] Graph is empty.")
        empty = gpd.GeoDataFrame(geometry=[], crs=target_crs)
        return empty, empty

    # ── 2. Node GDF + DEM elevation + smoothing ─────────────
    node_records = [
        {"x": n[0], "y": n[1], "geometry": Point(n)} for n in G.nodes
    ]
    nodes_gdf = gpd.GeoDataFrame(node_records, geometry="geometry", crs=target_crs)
    nodes_gdf = sample_dem(nodes_gdf, dem_tif, target_crs, output_col="ground_elev_m")
    nodes_gdf = smooth_elevations(nodes_gdf, radius_m=DEM_SMOOTHING_RADIUS_M)

    elev = {
        (round(r["x"], 3), round(r["y"], 3)): r["ground_elev_used_m"]
        for _, r in nodes_gdf.iterrows()
    }

    # ── 3. Orient each component toward WWTP2 ───────────────
    def nearest_node(nodes, point):
        arr = np.array(list(nodes))
        _, idx = cKDTree(arr).query([point.x, point.y])
        return tuple(arr[idx])

    component_roots = {}
    for comp_id, comp_nodes in enumerate(nx.connected_components(G)):
        sub  = G.subgraph(comp_nodes)
        root = nearest_node(comp_nodes, wwtp2_point)
        dist = nx.single_source_dijkstra_path_length(sub, root, weight="length_m")
        for nd in comp_nodes:
            G.nodes[nd]["comp_id"]        = comp_id
            G.nodes[nd]["dist_to_outlet"] = dist.get(nd, np.nan)
        component_roots[comp_id] = root

    # ── 4. Cumulative invert tracking — raw candidate detection
    raw_records     = []
    candidate_ctr   = 0

    for comp_id, comp_nodes in enumerate(nx.connected_components(G)):
        sub  = G.subgraph(comp_nodes)
        root = component_roots[comp_id]
        leaves = [n for n in sub.nodes if sub.degree(n) == 1 and n != root]
        if not leaves:
            leaves = [max(sub.nodes, key=lambda n: G.nodes[n].get("dist_to_outlet", 0))]

        for leaf_idx, leaf in enumerate(leaves):
            try:
                path = nx.shortest_path(sub, source=leaf, target=root, weight="length_m")
            except nx.NetworkXNoPath:
                continue

            invert_current = None
            cumulative_m   = 0.0
            last_raw_ps_m  = -p["min_path_dist_between_raw_m"]

            for u, v in zip(path[:-1], path[1:]):
                length_m = G[u][v].get("length_m", 1.0)
                z_u = elev.get(u, np.nan)
                z_v = elev.get(v, np.nan)
                cumulative_m += length_m

                if np.isnan(z_u) or np.isnan(z_v):
                    continue
                if invert_current is None:
                    invert_current = z_u - MIN_COVER_M

                invert_v_grav = invert_current - S_MIN * length_m
                cover_v_grav  = z_v - invert_v_grav
                depth_excess  = cover_v_grav - max_gravity_cover_m

                enough_excess = depth_excess >= p["min_depth_excess_raw_m"]
                far_enough    = (cumulative_m - last_raw_ps_m) >= p["min_path_dist_between_raw_m"]

                if enough_excess and far_enough:
                    candidate_ctr += 1
                    nat_slope    = (z_u - z_v) / length_m if length_m > 0 else np.nan
                    adverse_rise = max(0.0, z_v - z_u)

                    reasons = ["max_gravity_depth_exceeded"]
                    if adverse_rise > 0:
                        reasons.append("terrain_rises_towards_outlet")
                    if not np.isnan(nat_slope) and nat_slope < S_MIN:
                        reasons.append("insufficient_natural_slope")

                    raw_records.append({
                        "raw_candidate_id":      candidate_ctr,
                        "component_id":          comp_id,
                        "leaf_idx":              leaf_idx,
                        "station_x":             float(u[0]),
                        "station_y":             float(u[1]),
                        "ground_elev_m":         float(z_u),
                        "next_ground_elev_m":    float(z_v),
                        "edge_length_m":         float(length_m),
                        "cum_path_m":            float(cumulative_m),
                        "required_cover_next_m": float(cover_v_grav),
                        "depth_excess_m":        float(depth_excess),
                        "natural_slope":         float(nat_slope) if not np.isnan(nat_slope) else None,
                        "min_required_slope":    float(S_MIN),
                        "adverse_rise_m":        float(adverse_rise),
                        "reason":                ";".join(reasons),
                        "geometry":              Point(u),
                    })

                    # Hydraulic reset after proposed pumping intervention.
                    invert_current = z_v - MIN_COVER_M
                    last_raw_ps_m  = cumulative_m
                else:
                    invert_current = invert_v_grav

    if not raw_records:
        print("    No raw candidates found.")
        empty = gpd.GeoDataFrame(geometry=[], crs=target_crs)
        return empty, empty

    raw_gdf = gpd.GeoDataFrame(raw_records, geometry="geometry", crs=target_crs)
    raw_gdf = add_candidate_local_terrain_stats(
        raw_gdf,
        nodes_gdf,
        radius_m=(existing_ps_signature or {}).get(
            "calibration_radius_m",
            EXISTING_PS_CALIBRATION_RADIUS_M,
        ),
    )

    # ── 5. Upstream pipe length per candidate ───────────────
    comp_nodes_map = {i: ns for i, ns in enumerate(nx.connected_components(G))}

    def upstream_len(comp_id, station_xy):
        sub = G.subgraph(comp_nodes_map.get(comp_id, set()))
        nd  = round_node(station_xy)
        try:
            d_station = G.nodes[nd].get("dist_to_outlet", 0)
        except Exception:
            return 0.0
        return sum(
            data.get("length_m", 0.0)
            for u2, v2, data in sub.edges(data=True)
            if max(G.nodes[u2].get("dist_to_outlet", 0),
                   G.nodes[v2].get("dist_to_outlet", 0)) >= d_station
        )

    raw_gdf["upstream_pipe_length_m"] = [
        upstream_len(int(r["component_id"]), (r["station_x"], r["station_y"]))
        for _, r in raw_gdf.iterrows()
    ]

    # Priority score for optional-cap selection.
    raw_gdf["candidate_priority_score"] = (
        raw_gdf["depth_excess_m"].clip(lower=0)
        * np.log1p(raw_gdf["upstream_pipe_length_m"].clip(lower=0) / 1000.0)
        + 0.50 * raw_gdf["adverse_rise_m"].fillna(0).clip(lower=0)
        + 0.10 * raw_gdf["local_relief_m"].fillna(0).clip(lower=0)
    )

    # Existing-like similarity diagnostic.
    sig = existing_ps_signature or {}
    sig_score_thr = float(sig.get("local_terrain_score_p50", np.nan))
    if sig.get("valid", False) and np.isfinite(sig_score_thr) and sig_score_thr > 0:
        raw_gdf["existing_like_terrain_score_ratio"] = raw_gdf["local_terrain_score"] / sig_score_thr
    else:
        raw_gdf["existing_like_terrain_score_ratio"] = np.nan

    # ── 6. Greedy spatial deduplication ─────────────────────
    # Highest-priority candidates are kept first; nearby weaker candidates are discarded.
    sorted_raw  = raw_gdf.sort_values(
        ["candidate_priority_score", "depth_excess_m", "upstream_pipe_length_m"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    kept, kept_coords = [], []

    for _, row in sorted_raw.iterrows():
        pt = np.array([row["station_x"], row["station_y"]])
        if any(np.linalg.norm(pt - ck) < p["min_station_spacing_m"] for ck in kept_coords):
            continue
        kept.append(row)
        kept_coords.append(pt)

    merged = gpd.GeoDataFrame(kept, geometry="geometry", crs=target_crs).reset_index(drop=True)

    # ── 7. Mandatory / optional / screened-out classification ─
    mandatory_flags = []
    mandatory_reasons = []
    for _, row in merged.iterrows():
        flag, reason = mandatory_candidate_reason(row, p, existing_ps_signature=existing_ps_signature)
        mandatory_flags.append(bool(flag))
        mandatory_reasons.append(reason)

    merged["is_mandatory_ps"] = mandatory_flags
    merged["mandatory_reason"] = mandatory_reasons

    def screen(row):
        if bool(row.get("is_mandatory_ps", False)):
            return "final_candidate", "mandatory;" + str(row.get("mandatory_reason", ""))
        if row["upstream_pipe_length_m"] < p["min_upstream_pipe_length_m"]:
            return "screened_out", f"optional_screened_out;upstream_pipe_length_m < {p['min_upstream_pipe_length_m']}"
        if row["depth_excess_m"] < p["min_depth_excess_final_m"]:
            return "screened_out", f"optional_screened_out;depth_excess_m < {p['min_depth_excess_final_m']}"
        return "final_candidate", "optional;passed_all_criteria"

    statuses, screen_reasons = zip(*[screen(r) for _, r in merged.iterrows()]) if len(merged) > 0 else ([], [])

    merged["screening_status"] = list(statuses)
    merged["screening_reason"] = list(screen_reasons)

    final = merged[merged["screening_status"] == "final_candidate"].copy().reset_index(drop=True)

    # ── 8. Apply cap ONLY to optional stations ───────────────
    mandatory = final[final["is_mandatory_ps"] == True].copy()
    optional = final[final["is_mandatory_ps"] != True].copy()

    optional_cap_applied = False
    optional_before_cap = int(len(optional))

    if max_final_candidates is not None and len(optional) > int(max_final_candidates):
        optional = optional.sort_values(
            ["candidate_priority_score", "depth_excess_m", "upstream_pipe_length_m"],
            ascending=[False, False, False],
        ).head(int(max_final_candidates)).copy().reset_index(drop=True)
        optional["screening_reason"] = optional["screening_reason"].astype(str) + ";optional_kept_by_proportional_existing_PS_cap"
        optional_cap_applied = True

    mandatory["selection_class"] = "mandatory"
    optional["selection_class"] = "optional_selected"

    final = pd.concat([mandatory, optional], ignore_index=True)
    if len(final) > 0:
        final = final.sort_values(
            ["is_mandatory_ps", "candidate_priority_score", "depth_excess_m", "upstream_pipe_length_m"],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

    final["station_id"] = [f"PS_NEW_{i+1:03d}" for i in range(len(final))]
    final["max_optional_candidates_cap"] = max_final_candidates if max_final_candidates is not None else np.nan
    final["optional_candidates_before_cap"] = optional_before_cap
    final["optional_cap_applied"] = bool(optional_cap_applied)
    final["mandatory_ps_count_in_scenario"] = int(len(mandatory))
    final["optional_ps_count_in_scenario"] = int(len(optional))
    final["selection_mode"] = "necessity_based_mandatory_plus_optional_cap"

    # Keep the old column name for backward compatibility with maps/reports.
    final["max_final_candidates_cap"] = final["max_optional_candidates_cap"]

    print(f"    Mandatory candidates retained: {len(mandatory)}")
    print(f"    Optional candidates before cap: {optional_before_cap}")
    if max_final_candidates is not None:
        print(f"    Optional cap: {int(max_final_candidates)} | optional retained: {len(optional)} | cap applied: {optional_cap_applied}")
    print(f"    Final candidates total: {len(final)}")

    return final, raw_gdf

print("Necessity-based detect_ps_candidates() defined.")


Necessity-based detect_ps_candidates() defined.


## BLOCK G — Classify Existing PS by WWTP Service Area

In [7]:
def classify_existing_ps(existing_ps_gdf, wwtp2_pipes_gdf, wwtp1_pipes_gdf=None,
                          snap_dist_m=SNAP_DISTANCE_EXISTING_PS_M):
    """
    Tag each existing PS as WWTP2, WWTP1, or outside_both
    based on distance to the final pipe layers.
    """
    if existing_ps_gdf is None or len(existing_ps_gdf) == 0:
        return existing_ps_gdf

    ps = existing_ps_gdf.copy()

    w2_union = wwtp2_pipes_gdf.to_crs(ps.crs).geometry.unary_union         if wwtp2_pipes_gdf is not None and len(wwtp2_pipes_gdf) > 0 else None
    w1_union = wwtp1_pipes_gdf.to_crs(ps.crs).geometry.unary_union         if wwtp1_pipes_gdf is not None and len(wwtp1_pipes_gdf) > 0 else None

    labels, dists = [], []
    for _, row in ps.iterrows():
        pt = row.geometry
        if pt is None or pt.is_empty:
            labels.append("unknown"); dists.append(np.nan); continue

        d2 = pt.distance(w2_union) if w2_union is not None else np.inf
        d1 = pt.distance(w1_union) if w1_union is not None else np.inf
        dists.append(round(d2, 1))

        if d2 <= snap_dist_m and d2 <= d1:
            labels.append("WWTP2")
        elif d1 <= snap_dist_m and d1 < d2:
            labels.append("WWTP1")
        elif d2 <= snap_dist_m:
            labels.append("WWTP2")
        elif d1 <= snap_dist_m:
            labels.append("WWTP1")
        else:
            labels.append("outside_both")

    ps["nearest_wwtp_area"]     = labels
    ps["dist_to_wwtp2_pipes_m"] = dists
    return ps

print("classify_existing_ps() defined.")


classify_existing_ps() defined.


## BLOCK H — Main Loop: Process All Scenarios

In [8]:
all_summaries = []
scenario_cost_summaries = []
existing_wwtp2_summaries = []

for sc in SCENARIOS:
    key    = sc["key"]
    label  = sc["label"]
    family = sc["algorithm_family"]
    gpkg   = sc["gpkg"]
    fp     = FAMILY_PARAMS[family]
    max_optional_ps_cap = scenario_new_ps_cap(fp, EXISTING_PS_COUNT_REFERENCE)

    print(f"\n{'='*60}")
    print(f"  {label}  [{family}]")
    print(f"{'='*60}")
    print(f"  GPKG: {gpkg}")
    print(f"  Optional NEW PS cap: {max_optional_ps_cap}  (existing PS reference count = {EXISTING_PS_COUNT_REFERENCE})")
    print("  Mandatory stations are NOT limited by this cap.")

    if not gpkg.exists():
        print("  [SKIP] File not found.")
        continue

    layers = list_layers(gpkg)
    print(f"  Layers: {layers}")

    # ── Load new pipes ───────────────────────────────────────
    new_pipes, new_pipes_lyr = read_layer_auto(
        gpkg,
        keywords=["new", "pipe"],
        geometry_types=["LineString", "MultiLineString"]
    )

    if new_pipes is None or len(new_pipes) == 0:
        # fallback: first line layer
        for lyr in layers:
            s = gpd.read_file(gpkg, layer=lyr, rows=5)
            if s.geom_type.dropna().isin(["LineString","MultiLineString"]).any():
                new_pipes = gpd.read_file(gpkg, layer=lyr)
                new_pipes_lyr = lyr
                print(f"  Fallback: using layer '{lyr}' as new pipes")
                break

    if new_pipes is None or len(new_pipes) == 0:
        print("  [SKIP] No pipe layer found.")
        continue

    new_pipes = new_pipes.to_crs(CRS_PROJECTED)
    print(f"  New pipes  ({new_pipes_lyr}): {len(new_pipes)} features")

    new_pipe_length_m = float(new_pipes.geometry.length.sum())
    new_pipe_length_km = new_pipe_length_m / 1000.0
    new_pipe_cost_eur = new_pipe_length_m * PIPE_COST_EUR_PER_M

    # ── Load WWTP1/WWTP2 existing pipe layers ────────────────
    wwtp2_existing, _ = read_layer_auto(gpkg, ["wwtp2","existing"],
                                         ["LineString","MultiLineString"])
    wwtp1_existing, _ = read_layer_auto(gpkg, ["wwtp1","existing"],
                                         ["LineString","MultiLineString"])

    if wwtp2_existing is not None:
        wwtp2_existing = wwtp2_existing.to_crs(CRS_PROJECTED)
        print(f"  WWTP2 existing pipes: {len(wwtp2_existing)} features")
    if wwtp1_existing is not None:
        wwtp1_existing = wwtp1_existing.to_crs(CRS_PROJECTED)
        print(f"  WWTP1 existing pipes: {len(wwtp1_existing)} features")

    # ── Existing-PS calibration signature ────────────────────
    # Existing pumping stations are used as calibration evidence for critical contexts.
    print("\n  Building existing-PS terrain/network calibration signature …")
    existing_ps_signature = build_existing_ps_signature(
        existing_ps_gdf = existing_ps,
        pipe_gdfs       = [g for g in [wwtp1_existing, wwtp2_existing] if g is not None],
        dem_tif         = DEM_TIF,
        target_crs      = CRS_PROJECTED,
        calibration_radius_m = EXISTING_PS_CALIBRATION_RADIUS_M,
    )

    # ── Detect new PS candidates ─────────────────────────────
    print(f"\n  Running necessity-based hydraulic detection  [{family} params] …")
    final_ps, raw_ps = detect_ps_candidates(
        pipes_gdf             = new_pipes,
        wwtp2_point           = WWTP2_POINT,
        dem_tif               = DEM_TIF,
        target_crs            = CRS_PROJECTED,
        family_params         = fp,
        max_gravity_cover_m   = MAX_GRAVITY_COVER_M,
        max_final_candidates  = max_optional_ps_cap,
        existing_ps_signature = existing_ps_signature,
    )

    mandatory_new_ps_count = int(final_ps["is_mandatory_ps"].sum()) if len(final_ps) and "is_mandatory_ps" in final_ps.columns else 0
    optional_new_ps_count = int(len(final_ps) - mandatory_new_ps_count)

    print(f"  → Raw candidates       : {len(raw_ps)}")
    print(f"  → Final candidates     : {len(final_ps)}")
    print(f"      mandatory retained : {mandatory_new_ps_count}")
    print(f"      optional selected  : {optional_new_ps_count}")

    # ── Classify existing PS ─────────────────────────────────
    ref_pipes = wwtp2_existing if wwtp2_existing is not None else new_pipes
    ps_classified = classify_existing_ps(existing_ps, ref_pipes, wwtp1_existing)

    ps_wwtp2 = ps_classified[ps_classified["nearest_wwtp_area"] == "WWTP2"].copy() if ps_classified is not None else None
    ps_wwtp1 = ps_classified[ps_classified["nearest_wwtp_area"] == "WWTP1"].copy() if ps_classified is not None else None

    existing_ps_wwtp2_count = int(len(ps_wwtp2)) if ps_wwtp2 is not None else 0
    existing_ps_wwtp1_count = int(len(ps_wwtp1)) if ps_wwtp1 is not None else 0

    if ps_wwtp2 is not None:
        print(f"  Existing PS → WWTP2: {len(ps_wwtp2)}")
    if ps_wwtp1 is not None:
        print(f"  Existing PS → WWTP1: {len(ps_wwtp1)}")

    new_ps_capex_eur = len(final_ps) * NEW_PUMPING_STATION_CAPEX_EUR
    total_with_new_ps_eur = new_pipe_cost_eur + new_ps_capex_eur

    scenario_cost_summaries.append({
        "scenario": key,
        "label": label,
        "algorithm_family": family,
        "selection_mode": "necessity_based_mandatory_plus_optional_cap",
        "new_pipe_length_km": new_pipe_length_km,
        "new_pipe_cost_eur": new_pipe_cost_eur,
        "new_pipe_cost_million_eur": new_pipe_cost_eur / 1e6,
        "new_ps_count": int(len(final_ps)),
        "mandatory_new_ps_count": mandatory_new_ps_count,
        "optional_new_ps_count": optional_new_ps_count,
        "new_ps_capex_eur": new_ps_capex_eur,
        "new_ps_capex_million_eur": new_ps_capex_eur / 1e6,
        "total_new_pipe_plus_new_ps_eur": total_with_new_ps_eur,
        "total_new_pipe_plus_new_ps_million_eur": total_with_new_ps_eur / 1e6,
        "existing_ps_reference_count": EXISTING_PS_COUNT_REFERENCE,
        "max_optional_ps_cap": int(max_optional_ps_cap),
        # Backward-compatible column name. This is now an optional cap, not a hard cap on all new stations.
        "max_new_ps_cap": int(max_optional_ps_cap),
        "cap_applies_to": "optional_candidates_only",
        "existing_ps_now_wwtp2_count": existing_ps_wwtp2_count,
        "existing_ps_wwtp1_count": existing_ps_wwtp1_count,
        "pipe_cost_eur_per_m": PIPE_COST_EUR_PER_M,
        "new_ps_unit_capex_eur": NEW_PUMPING_STATION_CAPEX_EUR,
        "existing_like_relief_p50_m": existing_ps_signature.get("local_relief_p50_m", np.nan),
        "existing_like_terrain_score_p50": existing_ps_signature.get("local_terrain_score_p50", np.nan),
    })

    # ── Save GeoPackage ──────────────────────────────────────
    out_gpkg = OUTPUT_DIR / f"{key}_pumping_stations.gpkg"
    if out_gpkg.exists():
        out_gpkg.unlink()

    write_layer(final_ps,       out_gpkg, "new_ps_final_candidates")
    write_layer(raw_ps,         out_gpkg, "new_ps_raw_candidates")

    if len(final_ps) > 0 and "is_mandatory_ps" in final_ps.columns:
        write_layer(final_ps[final_ps["is_mandatory_ps"] == True].copy(), out_gpkg, "new_ps_mandatory_candidates")
        write_layer(final_ps[final_ps["is_mandatory_ps"] != True].copy(), out_gpkg, "new_ps_optional_selected")

    write_layer(new_pipes,      out_gpkg, "new_pipes_reference")
    write_layer(ps_wwtp2,       out_gpkg, "existing_ps_now_wwtp2")
    write_layer(ps_wwtp1,       out_gpkg, "existing_ps_wwtp1")
    write_layer(ps_classified,  out_gpkg, "existing_ps_all_classified")

    wwtp2_gdf = gpd.GeoDataFrame(
        [{"name":"WWTP2", "lon":WWTP2_LON, "lat":WWTP2_LAT, "x_projected": wx, "y_projected": wy, "geometry": WWTP2_POINT}],
        geometry="geometry", crs=CRS_PROJECTED
    )
    write_layer(wwtp2_gdf, out_gpkg, "wwtp2_point")
    print(f"  Saved: {out_gpkg.name}")

    # ── Collect summary rows: NEW proposed PS ────────────────
    for _, row in final_ps.iterrows():
        all_summaries.append({
            "scenario":               key,
            "label":                  label,
            "algorithm_family":       family,
            "selection_mode":         row.get("selection_mode", "necessity_based_mandatory_plus_optional_cap"),
            "station_id":             row.get("station_id",""),
            "type":                   "NEW – proposed",
            "selection_class":         row.get("selection_class", ""),
            "is_mandatory_ps":        row.get("is_mandatory_ps", False),
            "mandatory_reason":        row.get("mandatory_reason", ""),
            "x_projected":            round(row.geometry.x, 1),
            "y_projected":            round(row.geometry.y, 1),
            "ground_elev_m":          row.get("ground_elev_m", np.nan),
            "depth_excess_m":         row.get("depth_excess_m", np.nan),
            "upstream_pipe_length_m": row.get("upstream_pipe_length_m", np.nan),
            "adverse_rise_m":         row.get("adverse_rise_m", np.nan),
            "local_relief_m":         row.get("local_relief_m", np.nan),
            "local_terrain_score":    row.get("local_terrain_score", np.nan),
            "existing_like_terrain_score_ratio": row.get("existing_like_terrain_score_ratio", np.nan),
            "candidate_priority_score": row.get("candidate_priority_score", np.nan),
            "unit_capex_eur":         NEW_PUMPING_STATION_CAPEX_EUR,
            "screening_reason":       row.get("screening_reason",""),
            "reason":                 row.get("reason",""),
        })

    # ── Collect summary rows: existing PS now serving WWTP2 ──
    if ps_wwtp2 is not None:
        name_col = next((c for c in ["Name","pump_station","name","id", "OBJECTID"] if c in ps_wwtp2.columns), None)
        for idx, row in ps_wwtp2.iterrows():
            station_name = row[name_col] if name_col else f"existing_PS_{idx}"
            existing_wwtp2_summaries.append({
                "scenario": key,
                "label": label,
                "algorithm_family": family,
                "existing_station_id": station_name,
                "x_projected": round(row.geometry.x, 1),
                "y_projected": round(row.geometry.y, 1),
                "dist_to_wwtp2_pipes_m": row.get("dist_to_wwtp2_pipes_m", np.nan),
                "dist_to_wwtp1_pipes_m": row.get("dist_to_wwtp1_pipes_m", np.nan),
                "source_file": row.get("ps_source_file", ""),
            })
            all_summaries.append({
                "scenario":               key,
                "label":                  label,
                "algorithm_family":       family,
                "selection_mode":         "existing_station_classification",
                "station_id":             station_name,
                "type":                   "EXISTING – now serves WWTP2",
                "selection_class":         "existing_asset",
                "is_mandatory_ps":        np.nan,
                "mandatory_reason":        "existing_asset_not_new_capex",
                "x_projected":            round(row.geometry.x, 1),
                "y_projected":            round(row.geometry.y, 1),
                "ground_elev_m":          np.nan,
                "depth_excess_m":         np.nan,
                "upstream_pipe_length_m": np.nan,
                "adverse_rise_m":         np.nan,
                "local_relief_m":         np.nan,
                "local_terrain_score":    np.nan,
                "existing_like_terrain_score_ratio": np.nan,
                "candidate_priority_score": np.nan,
                "unit_capex_eur":         0.0,
                "screening_reason":       f"dist_to_wwtp2_pipes={row.get('dist_to_wwtp2_pipes_m','?')} m",
                "reason":                 "existing station in WWTP2 service area",
            })

print("\n✓ All scenarios processed.")



  Rooted Dijkstra  [non_budgeted]
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\After_Manual_Networks_Correction\dijkstra_final_manual_correction\final_manual_corrected_network.gpkg
  Optional NEW PS cap: 17  (existing PS reference count = 22)
  Mandatory stations are NOT limited by this cap.
  Layers: ['existing_pipes_WWTP1_final', 'existing_pipes_WWTP2_final_manual', 'new_pipes_WWTP2_final', 'all_existing_pipes_reference', 'demand_nodes_final_manual_assignment', 'demand_nodes_preserved_upstream_assignment', 'demand_nodes_considered_assigned_WWTP1', 'demand_nodes_considered_assigned_WWTP2', 'demand_nodes_not_considered_excluded', 'municipal_boundaries', 'pumping_stations']
  New pipes  (new_pipes_WWTP2_final): 16738 features
  WWTP2 existing pipes: 2695 features
  WWTP1 existing pipes: 3713 features

  Building existing-PS terrain/network calibration signature …
    DEM sampled 22 points — missing: 0.0%
    DEM sampled 6461 points — missing: 0.0%
    [Calibration] Ex

## BLOCK I — Summary CSV & Console Report

In [9]:
print("\n" + "="*80)
print("  GLOBAL PUMPING-STATION SUMMARY")
print("="*80)

from IPython.display import display

if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    csv_path   = OUTPUT_DIR / "pumping_stations_summary_all_scenarios.csv"
    summary_df.to_csv(csv_path, index=False)
    print(f"\nSummary CSV: {csv_path}")
    print(f"Total station-summary rows: {len(summary_df)}")

    # ── Scenario cost summary ───────────────────────────────
    cost_df = pd.DataFrame(scenario_cost_summaries)
    cost_csv = OUTPUT_DIR / "pumping_stations_cost_summary_by_scenario.csv"
    cost_df.to_csv(cost_csv, index=False)

    print("\n── COST SUMMARY BY SCENARIO ─────────────────────────────")
    cost_display_cols = [
        "scenario", "algorithm_family", "selection_mode",
        "new_pipe_length_km",
        "new_pipe_cost_million_eur",
        "new_ps_count",
        "mandatory_new_ps_count",
        "optional_new_ps_count",
        "new_ps_capex_million_eur",
        "total_new_pipe_plus_new_ps_million_eur",
        "existing_ps_reference_count",
        "max_optional_ps_cap",
        "cap_applies_to",
        "existing_ps_now_wwtp2_count",
    ]
    display(cost_df[[c for c in cost_display_cols if c in cost_df.columns]].round(3))
    print(f"Cost summary CSV: {cost_csv}")

    # ── NEW proposed PS per scenario ────────────────────────
    new_ps = summary_df[summary_df["type"] == "NEW – proposed"].copy()
    print("\n── NEW proposed pumping stations per scenario ──────────")
    if len(new_ps) == 0:
        print("  No new pumping stations proposed.")
    else:
        for key_tuple, grp in new_ps.groupby(["scenario","algorithm_family"]):
            sc_key, fam = key_tuple
            n_mand = int(grp["is_mandatory_ps"].fillna(False).astype(bool).sum()) if "is_mandatory_ps" in grp.columns else 0
            n_opt = int(len(grp) - n_mand)
            print(f"  {sc_key:30s} [{fam:12s}]  {len(grp)} new station(s)  | mandatory={n_mand}, optional={n_opt}")
            for _, r in grp.sort_values(["is_mandatory_ps", "candidate_priority_score"], ascending=[False, False]).iterrows():
                elev_str = f"  elev={r['ground_elev_m']:.1f}m" if pd.notna(r['ground_elev_m']) else ""
                excess_str = f"  depth_excess={r['depth_excess_m']:.2f}m" if pd.notna(r['depth_excess_m']) else ""
                up_str = f"  upstream={r['upstream_pipe_length_m']:.0f}m" if pd.notna(r['upstream_pipe_length_m']) else ""
                cls_str = f"  class={r.get('selection_class','')}"
                print(f"      {r['station_id']}{cls_str}{elev_str}{excess_str}{up_str}")
                print(f"        reason: {r['reason']}")
                print(f"        screening: {r['screening_reason']}")
                if bool(r.get("is_mandatory_ps", False)):
                    print(f"        mandatory reason: {r.get('mandatory_reason','')}")

    # ── EXISTING PS now serving WWTP2 ───────────────────────
    exist_ps = summary_df[summary_df["type"] == "EXISTING – now serves WWTP2"].copy()
    existing_csv = OUTPUT_DIR / "existing_pumping_stations_now_serving_WWTP2.csv"

    if existing_wwtp2_summaries:
        existing_wwtp2_df = pd.DataFrame(existing_wwtp2_summaries)
        existing_wwtp2_df.to_csv(existing_csv, index=False)
    else:
        existing_wwtp2_df = pd.DataFrame()
        existing_wwtp2_df.to_csv(existing_csv, index=False)

    print("\n── EXISTING pumping stations now in WWTP2 service area ──")
    if len(exist_ps) == 0:
        print("  No existing pumping stations classified as WWTP2.")
    else:
        for key_tuple, grp in exist_ps.groupby(["scenario","algorithm_family"]):
            sc_key, fam = key_tuple
            print(f"  {sc_key:30s} [{fam:12s}]  {len(grp)} existing station(s)")
            for _, r in grp.iterrows():
                print(f"      → {r['station_id']}  ({r['screening_reason']})")
        print(f"Existing-WWTP2 summary CSV: {existing_csv}")

    # ── Full table display ──────────────────────────────────
    print("\n── FULL STATION SUMMARY TABLE ───────────────────────────")
    display_cols = [
        "scenario", "algorithm_family", "station_id", "type", "selection_class",
        "is_mandatory_ps", "depth_excess_m", "upstream_pipe_length_m",
        "adverse_rise_m", "local_relief_m", "existing_like_terrain_score_ratio",
        "candidate_priority_score", "unit_capex_eur", "screening_reason", "mandatory_reason", "reason",
    ]
    display(summary_df[[c for c in display_cols if c in summary_df.columns]].round(3))

    # ── Interpretation note ─────────────────────────────────
    print("\nInterpretation:")
    print("  - New pipe cost uses PIPE_COST_EUR_PER_M =", PIPE_COST_EUR_PER_M, "€/m.")
    print("  - New pumping-station CAPEX uses NEW_PUMPING_STATION_CAPEX_EUR =", NEW_PUMPING_STATION_CAPEX_EUR, "€/station.")
    print("  - Existing PS classified as WWTP2 are not counted as new CAPEX here; they are existing assets now associated with WWTP2 service area.")
    print("  - The proportional cap is now applied only to optional candidates.")
    print("  - Mandatory candidates are retained even if the final number exceeds the family cap.")
    print("  - Mandatory classification is based on severe depth excess, adverse terrain, upstream network importance, and existing-PS-like terrain/network context.")

else:
    print("No pumping stations detected across any scenario.")

print(f"\nAll outputs in:\n{OUTPUT_DIR}")



  GLOBAL PUMPING-STATION SUMMARY

Summary CSV: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\pumping_stations_summary_all_scenarios.csv
Total station-summary rows: 183

── COST SUMMARY BY SCENARIO ─────────────────────────────


,scenario,algorithm_family,selection_mode,new_pipe_length_km,new_pipe_cost_million_eur,new_ps_count,mandatory_new_ps_count,optional_new_ps_count,new_ps_capex_million_eur,total_new_pipe_plus_new_ps_million_eur,existing_ps_reference_count,max_optional_ps_cap,cap_applies_to,existing_ps_now_wwtp2_count
0,dijkstra,non_budgeted,necessity_based_mandatory_plus_optional_cap,525.765,210.306,30,29,1,45.0,255.306,22,17,optional_candidates_only,13
1,prim_steiner,non_budgeted,necessity_based_mandatory_plus_optional_cap,414.699,165.880,29,28,1,43.5,209.380,22,17,optional_candidates_only,13
2,shared_trunk,budgeted,necessity_based_mandatory_plus_optional_cap,249.993,99.997,20,19,1,30.0,129.997,22,8,optional_candidates_only,13
3,urban_priority,budgeted,necessity_based_mandatory_plus_optional_cap,249.928,99.971,20,20,0,30.0,129.971,22,8,optional_candidates_only,13
4,municipality_targets,budgeted,necessity_based_mandatory_plus_optional_cap,249.964,99.986,19,19,0,28.5,128.486,22,8,optional_candidates_only,13


Cost summary CSV: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\pumping_stations_cost_summary_by_scenario.csv

── NEW proposed pumping stations per scenario ──────────
  dijkstra                       [non_budgeted]  30 new station(s)  | mandatory=29, optional=1
      PS_NEW_001  class=mandatory  elev=108.6m  depth_excess=51.79m  upstream=409653m
        reason: max_gravity_depth_exceeded
        screening: mandatory;mandatory_high_depth_excess>= 5.0m;mandatory_extreme_depth_excess>= 7.0m;mandatory_existing_PS_like_terrain_network_signature;mandatory_severe_existing_PS_like_context
        mandatory reason: mandatory_high_depth_excess>= 5.0m;mandatory_extreme_depth_excess>= 7.0m;mandatory_existing_PS_like_terrain_network_signature;mandatory_severe_existing_PS_like_context
      PS_NEW_002  class=mandatory  elev=114.8m  depth_excess=53.78m  upstream=332642m
        reason: max_gravity_depth_exceeded;terrain_rises_towards_outlet;insufficient_natural_s

,scenario,algorithm_family,station_id,type,selection_class,is_mandatory_ps,depth_excess_m,upstream_pipe_length_m,adverse_rise_m,local_relief_m,existing_like_terrain_score_ratio,candidate_priority_score,unit_capex_eur,screening_reason,mandatory_reason,reason
0,dijkstra,non_budgeted,PS_NEW_001,NEW – proposed,mandatory,True,51.791,409653.020,0.000,109.880,11.297,322.650,1500000.0,mandatory;mandatory_high_depth_excess>= 5.0m;m...,mandatory_high_depth_excess>= 5.0m;mandatory_e...,max_gravity_depth_exceeded
1,dijkstra,non_budgeted,PS_NEW_002,NEW – proposed,mandatory,True,53.784,332641.800,3.236,80.009,4.642,322.107,1500000.0,mandatory;mandatory_high_depth_excess>= 5.0m;m...,mandatory_high_depth_excess>= 5.0m;mandatory_e...,max_gravity_depth_exceeded;terrain_rises_towar...
2,dijkstra,non_budgeted,PS_NEW_003,NEW – proposed,mandatory,True,34.147,466598.205,13.043,62.490,3.385,222.691,1500000.0,mandatory;mandatory_high_depth_excess>= 5.0m;m...,mandatory_high_depth_excess>= 5.0m;mandatory_e...,max_gravity_depth_exceeded;terrain_rises_towar...
3,dijkstra,non_budgeted,PS_NEW_004,NEW – proposed,mandatory,True,21.480,113982.428,26.957,69.838,6.751,122.381,1500000.0,mandatory;mandatory_high_depth_excess>= 5.0m;m...,mandatory_high_depth_excess>= 5.0m;mandatory_e...,max_gravity_depth_exceeded;terrain_rises_towar...
4,dijkstra,non_budgeted,PS_NEW_005,NEW – proposed,mandatory,True,16.501,491538.855,15.300,89.461,5.960,118.894,1500000.0,mandatory;mandatory_high_depth_excess>= 5.0m;m...,mandatory_high_depth_excess>= 5.0m;mandatory_e...,max_gravity_depth_exceeded;terrain_rises_towar...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,municipality_targets,budgeted,Pumping_Station_HA,EXISTING – now serves WWTP2,existing_asset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,dist_to_wwtp2_pipes=0.0 m,existing_asset_not_new_capex,existing station in WWTP2 service area
179,municipality_targets,budgeted,Pumping_Station_R2,EXISTING – now serves WWTP2,existing_asset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,dist_to_wwtp2_pipes=0.0 m,existing_asset_not_new_capex,existing station in WWTP2 service area
180,municipality_targets,budgeted,Pumping_Station_X,EXISTING – now serves WWTP2,existing_asset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,dist_to_wwtp2_pipes=0.0 m,existing_asset_not_new_capex,existing station in WWTP2 service area
181,municipality_targets,budgeted,Pumping_Station_XA,EXISTING – now serves WWTP2,existing_asset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,dist_to_wwtp2_pipes=4.6 m,existing_asset_not_new_capex,existing station in WWTP2 service area



Interpretation:
  - New pipe cost uses PIPE_COST_EUR_PER_M = 400.0 €/m.
  - New pumping-station CAPEX uses NEW_PUMPING_STATION_CAPEX_EUR = 1500000.0 €/station.
  - Existing PS classified as WWTP2 are not counted as new CAPEX here; they are existing assets now associated with WWTP2 service area.
  - The proportional cap is now applied only to optional candidates.
  - Mandatory candidates are retained even if the final number exceeds the family cap.
  - Mandatory classification is based on severe depth excess, adverse terrain, upstream network importance, and existing-PS-like terrain/network context.

All outputs in:
C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning


In [10]:
# ============================================================
# BLOCK 9 -- COMPLETE INTERACTIVE MAPS WITH ALL PIPES
# ============================================================
# Purpose:
#   Create HTML maps where pumping stations are shown as individual points
#   and ALL final network pipes are shown.
#
# It reads:
#   1. Pumping-station outputs:
#      Results/PunpingStations_Positioning/<scenario>_pumping_stations.gpkg
#
#   2. Final corrected network outputs:
#      Results/After_Manual_Networks_Correction/<scenario>_final_manual_correction/final_manual_corrected_network.gpkg
#
# Outputs:
#   Results/PunpingStations_Positioning/interactive_maps_complete_network/
#     <scenario>_COMPLETE_network_pumping_stations_map.html
#     ALL_SCENARIOS_COMPLETE_network_pumping_stations_overview.html
#     complete_network_map_layer_counts.csv

print("=" * 110)
print("BLOCK 9C -- COMPLETE INTERACTIVE MAPS WITH ALL PIPES")
print("=" * 110)

import sys
import subprocess
import importlib.util
from pathlib import Path

if importlib.util.find_spec("folium") is None:
    print("Installing folium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])

import folium
from folium.plugins import Fullscreen, MeasureControl, MiniMap
import pandas as pd
import geopandas as gpd
import fiona

# ------------------------------------------------------------
# 9C.1 Output folder
# ------------------------------------------------------------
MAP_DIR_COMPLETE = OUTPUT_DIR / "interactive_maps_complete_network"
MAP_DIR_COMPLETE.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 9C.2 Corrected-network base folder
# ------------------------------------------------------------
# This is where the final_manual_corrected_network.gpkg files are saved.
if "MANUAL_CORRECTION_DIR" in globals():
    CORRECTED_NETWORK_BASE = Path(MANUAL_CORRECTION_DIR)
else:
    CORRECTED_NETWORK_BASE = Path(
        r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\After_Manual_Networks_Correction"
    )

# ------------------------------------------------------------
# 9C.3 Display settings
# ------------------------------------------------------------
SHOW_RAW_CANDIDATES = False
SHOW_EXISTING_WWTP1 = True
SHOW_ALL_PIPES = True
SIMPLIFY_PIPES_TOLERANCE_M = 5.0

# ------------------------------------------------------------
# 9C.4 Colors
# ------------------------------------------------------------
# Point colors = pumping stations / WWTP
# Line colors  = pipes
#
# The point colors are intentionally different from the pipe colors
# to avoid visual confusion on the HTML maps.

# Pumping station / point colors
COLOR_NEW_PS = "#ff00ff"                    # magenta - new proposed pumping stations
COLOR_EXISTING_PS_NOW_WWTP2 = "#ffd700"    # yellow/gold - existing PS now assigned to WWTP2
COLOR_EXISTING_PS_WWTP1 = "#ff7f00"        # orange - existing PS remaining WWTP1
COLOR_RAW_CANDIDATE = "#8b4513"            # brown - raw candidate pumping stations
COLOR_WWTP2 = "#000000"                    # black - WWTP2 point

# Pipe / line colors
COLOR_EXISTING_PIPES_WWTP1 = "#1a9850"     # green line - existing pipes WWTP1
COLOR_EXISTING_PIPES_WWTP2 = "#2b6cb0"     # blue line - existing pipes WWTP2
COLOR_NEW_PIPES_WWTP1 = "#984ea3"          # purple line - new pipes WWTP1
COLOR_NEW_PIPES_WWTP2 = "#e41a1c"          # red line - new pipes WWTP2
COLOR_OTHER_PIPES = "#636363"              # grey line - fallback / other pipes
# ------------------------------------------------------------
# 9C.5 Helper functions
# ------------------------------------------------------------
def _scenario_key(sc):
    if isinstance(sc, dict):
        return sc.get("key", sc.get("scenario_key", sc.get("scenario", "UNKNOWN")))
    return str(sc)


def _scenario_label(sc):
    if isinstance(sc, dict):
        return sc.get("label", sc.get("display_name", _scenario_key(sc)))
    return str(sc)


def _scenario_family(sc):
    if isinstance(sc, dict):
        return sc.get("algorithm_family", "UNKNOWN")
    return "UNKNOWN"


def _scenario_corrected_dir_name(key):
    mapping = {
        "dijkstra": "dijkstra_final_manual_correction",
        "prim_steiner": "prim_steiner_final_manual_correction",
        "shared_trunk": "shared_trunk_final_manual_correction",
        "urban_priority": "urban_priority_final_manual_correction",
        "municipality_targets": "municipality_targets_final_manual_correction",
    }
    return mapping.get(key, f"{key}_final_manual_correction")


def _layer_exists(gpkg, layer):
    try:
        return layer in fiona.listlayers(str(gpkg))
    except Exception:
        return False


def _safe_read(gpkg, layer, target_crs=CRS_PROJECTED):
    gpkg = Path(gpkg)

    if not gpkg.exists() or not _layer_exists(gpkg, layer):
        return gpd.GeoDataFrame(geometry=[], crs=target_crs)

    try:
        gdf = gpd.read_file(gpkg, layer=layer)

        if gdf.crs is None:
            gdf = gdf.set_crs(target_crs)
        else:
            gdf = gdf.to_crs(target_crs)

        gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
        return gdf

    except Exception as exc:
        print(f"  [WARNING] Could not read {layer} from {gpkg}: {exc}")
        return gpd.GeoDataFrame(geometry=[], crs=target_crs)


def _read_first_existing_layer(gpkg, layer_candidates, target_crs=CRS_PROJECTED):
    for layer in layer_candidates:
        if _layer_exists(gpkg, layer):
            gdf = _safe_read(gpkg, layer, target_crs=target_crs)
            if len(gdf) > 0:
                return gdf, layer

    return gpd.GeoDataFrame(geometry=[], crs=target_crs), None


def _normalize_text_series(s):
    return (
        s.astype(str)
        .str.upper()
        .str.replace("_", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace(" ", "", regex=False)
    )


def _empty_like_existing(gdf):
    if gdf is not None and hasattr(gdf, "crs"):
        return gpd.GeoDataFrame(geometry=[], crs=gdf.crs)
    return gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED)


def _filter_existing_ps_by_wwtp(existing_all, target="WWTP2"):
    """
    Correctly split existing pumping stations into WWTP1 / WWTP2.
    """
    if existing_all is None or len(existing_all) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED)

    g = existing_all.copy()
    target_norm = str(target).upper().replace("_", "").replace("-", "").replace(" ", "")

    if "nearest_wwtp_area" in g.columns:
        area = _normalize_text_series(g["nearest_wwtp_area"])

        if target_norm == "WWTP2":
            mask = (
                area.str.contains("WWTP2", na=False)
                | area.str.contains("WWTPNORTH", na=False)
                | area.str.contains("NORTH", na=False)
                | area.str.contains("2NORTH", na=False)
            )
        elif target_norm == "WWTP1":
            mask = (
                area.str.contains("WWTP1", na=False)
                | area.str.contains("WWTPSOUTH", na=False)
                | area.str.contains("SOUTH", na=False)
                | area.str.contains("1SOUTH", na=False)
            )
        else:
            mask = pd.Series(False, index=g.index)

        return g[mask].copy()

    if "final_wwtp_assignment" in g.columns:
        area = _normalize_text_series(g["final_wwtp_assignment"])

        if target_norm == "WWTP2":
            mask = area.str.contains("WWTP2", na=False) | area.str.contains("NORTH", na=False)
        elif target_norm == "WWTP1":
            mask = area.str.contains("WWTP1", na=False) | area.str.contains("SOUTH", na=False)
        else:
            mask = pd.Series(False, index=g.index)

        return g[mask].copy()

    for col in ["wwtp_assignment", "assigned_wwtp", "assignment", "service_area"]:
        if col in g.columns:
            area = _normalize_text_series(g[col])

            if target_norm == "WWTP2":
                mask = area.str.contains("WWTP2", na=False) | area.str.contains("NORTH", na=False)
            elif target_norm == "WWTP1":
                mask = area.str.contains("WWTP1", na=False) | area.str.contains("SOUTH", na=False)
            else:
                mask = pd.Series(False, index=g.index)

            return g[mask].copy()

    if (
        "dist_to_wwtp2_pipes_m" in g.columns
        and "dist_to_wwtp1_pipes_m" in g.columns
    ):
        d2 = pd.to_numeric(g["dist_to_wwtp2_pipes_m"], errors="coerce")
        d1 = pd.to_numeric(g["dist_to_wwtp1_pipes_m"], errors="coerce")

        snap_distance = globals().get("SNAP_DISTANCE_EXISTING_PS_M", 500.0)

        if target_norm == "WWTP2":
            mask = (
                d2.notna()
                & (d2 <= snap_distance)
                & (d1.isna() | (d2 <= d1))
            )
        elif target_norm == "WWTP1":
            mask = (
                d1.notna()
                & (d1 <= snap_distance)
                & (d2.isna() | (d1 < d2))
            )
        else:
            mask = pd.Series(False, index=g.index)

        return g[mask].copy()

    print("  [WARNING] Could not classify existing pumping stations by WWTP.")
    return _empty_like_existing(g)


def _to_wgs(gdf):
    if gdf is None or len(gdf) == 0:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_PROJECTED)

    return gdf.to_crs("EPSG:4326")


def _popup_html(row, fields):
    parts = []

    for f in fields:
        if f in row.index:
            val = row[f]

            if pd.isna(val):
                continue

            if isinstance(val, float):
                val = round(val, 3)

            parts.append(f"<b>{f}</b>: {val}")

    return "<br>".join(parts) if parts else "No attributes"


def _add_point_layer_no_cluster(
    m,
    gdf,
    layer_name,
    marker_color,
    fill_color=None,
    radius=5,
    fields=None,
    show=True,
):
    if gdf is None or len(gdf) == 0:
        return 0

    fill_color = fill_color or marker_color
    fields = fields or []
    g = _to_wgs(gdf)

    fg = folium.FeatureGroup(
        name=f"{layer_name} ({len(g)})",
        show=show,
    )

    for _, row in g.iterrows():
        geom = row.geometry

        if geom is None or geom.is_empty:
            continue

        if geom.geom_type != "Point":
            geom = geom.centroid

        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=radius,
            color=marker_color,
            fill=True,
            fill_color=fill_color,
            fill_opacity=0.90,
            weight=2,
            popup=folium.Popup(_popup_html(row, fields), max_width=450),
        ).add_to(fg)

    fg.add_to(m)
    return len(g)


def _add_line_layer(
    m,
    gdf,
    layer_name,
    color,
    weight=2,
    opacity=0.70,
    show=True,
):
    if gdf is None or len(gdf) == 0:
        return 0

    g = gdf.copy()

    if SIMPLIFY_PIPES_TOLERANCE_M and SIMPLIFY_PIPES_TOLERANCE_M > 0:
        g["geometry"] = g.geometry.simplify(
            SIMPLIFY_PIPES_TOLERANCE_M,
            preserve_topology=True,
        )

    g = _to_wgs(g)

    tooltip_fields = [
        c for c in [
            "scenario",
            "pipe_group",
            "network_type",
            "final_wwtp_assignment",
            "length_m",
            "length_km",
            "municipality",
        ]
        if c in g.columns
    ]

    folium.GeoJson(
        data=g.__geo_interface__,
        name=f"{layer_name} ({len(g)})",
        show=show,
        style_function=lambda feature, color=color, weight=weight, opacity=opacity: {
            "color": color,
            "weight": weight,
            "opacity": opacity,
        },
        tooltip=folium.GeoJsonTooltip(fields=tooltip_fields) if tooltip_fields else None,
    ).add_to(m)

    return len(g)


def _map_center_from_layers(layers, fallback_point=None):
    valid = [g for g in layers if g is not None and len(g) > 0]

    if valid:
        geoms = []

        for g in valid:
            gw = _to_wgs(g)
            if len(gw) > 0:
                geoms.append(gw.geometry.to_frame("geometry"))

        if len(geoms) > 0:
            union = pd.concat(geoms, ignore_index=True)
            ug = gpd.GeoDataFrame(union, geometry="geometry", crs="EPSG:4326")

            minx, miny, maxx, maxy = ug.total_bounds
            center = [(miny + maxy) / 2.0, (minx + maxx) / 2.0]
            bounds = [[miny, minx], [maxy, maxx]]
            return center, bounds

    if fallback_point is not None:
        p = (
            gpd.GeoDataFrame(geometry=[fallback_point], crs=CRS_PROJECTED)
            .to_crs("EPSG:4326")
            .geometry.iloc[0]
        )
        return [p.y, p.x], None

    return [34.95, 33.60], None


def _add_map_controls(m):
    folium.LayerControl(collapsed=False).add_to(m)
    Fullscreen(position="topleft").add_to(m)
    MeasureControl(position="bottomleft", primary_length_unit="meters").add_to(m)
    MiniMap(toggle_display=True).add_to(m)


def _add_complete_legend(m, title="Complete Network Legend"):
    legend_html = f"""
    <div style='position: fixed; bottom: 30px; left: 30px; z-index:9999;
                background-color: white; padding: 10px; border: 2px solid #444;
                font-size: 13px; max-width: 360px;'>
      <b>{title}</b><br>
      <span style='color:{COLOR_NEW_PS};'>●</span> New proposed pumping station<br>
      <span style='color:{COLOR_EXISTING_PS_NOW_WWTP2};'>●</span> Existing pumping station now assigned to WWTP2<br>
      <span style='color:{COLOR_EXISTING_PS_WWTP1};'>●</span> Existing pumping station remaining WWTP1<br>
      <span style='color:{COLOR_RAW_CANDIDATE};'>●</span> Raw candidate pumping station<br>
      <span style='color:{COLOR_WWTP2};'>●</span> WWTP2<br>
      <hr style='margin:4px 0;'>
      <span style='color:{COLOR_EXISTING_PIPES_WWTP1};'>━</span> Existing pipes WWTP1<br>
      <span style='color:{COLOR_EXISTING_PIPES_WWTP2};'>━</span> Existing pipes WWTP2<br>
      <span style='color:{COLOR_NEW_PIPES_WWTP1};'>━</span> New pipes WWTP1<br>
      <span style='color:{COLOR_NEW_PIPES_WWTP2};'>━</span> New pipes WWTP2<br>
      <span style='color:{COLOR_OTHER_PIPES};'>━</span> Other / fallback pipes<br>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))


def _read_all_final_pipe_layers(corrected_gpkg):
    """
    Reads all relevant pipe layers from the final corrected network GeoPackage.
    The exact layer names can differ slightly, so this function is robust.
    """

    pipe_layers = {}

    existing_wwtp1, layer = _read_first_existing_layer(
        corrected_gpkg,
        [
            "existing_pipes_WWTP1_final",
            "existing_pipes_WWTP1_final_manual",
            "existing_pipes_wwtp1_final",
            "existing_pipes_WWTP1",
        ],
    )
    pipe_layers["existing_wwtp1"] = (existing_wwtp1, layer)

    existing_wwtp2, layer = _read_first_existing_layer(
        corrected_gpkg,
        [
            "existing_pipes_WWTP2_final_manual",
            "existing_pipes_WWTP2_final",
            "existing_pipes_wwtp2_final",
            "existing_pipes_WWTP2",
        ],
    )
    pipe_layers["existing_wwtp2"] = (existing_wwtp2, layer)

    new_wwtp1, layer = _read_first_existing_layer(
        corrected_gpkg,
        [
            "new_pipes_WWTP1_final",
            "new_pipes_WWTP1_final_manual",
            "new_pipes_wwtp1_final",
            "new_pipes_WWTP1",
        ],
    )
    pipe_layers["new_wwtp1"] = (new_wwtp1, layer)

    new_wwtp2, layer = _read_first_existing_layer(
        corrected_gpkg,
        [
            "new_pipes_WWTP2_final",
            "new_pipes_WWTP2_final_manual",
            "new_pipes_wwtp2_final",
            "new_pipes_WWTP2",
            "new_pipes_reference",
        ],
    )
    pipe_layers["new_wwtp2"] = (new_wwtp2, layer)

    # Optional fallback combined layers, only used if individual layers are missing.
    combined_final, layer = _read_first_existing_layer(
        corrected_gpkg,
        [
            "final_pipes",
            "all_final_pipes",
            "final_manual_corrected_pipes",
            "pipes_final_manual_assignment",
            "network_edges_final",
        ],
    )
    pipe_layers["combined_final"] = (combined_final, layer)

    return pipe_layers


map_rows = []
overview_layers = []
overview_pipe_layers = []

# ------------------------------------------------------------
# 9C.6 Scenario maps
# ------------------------------------------------------------
for sc in SCENARIOS:
    key = _scenario_key(sc)
    label = _scenario_label(sc)
    family = _scenario_family(sc)

    pumping_gpkg = OUTPUT_DIR / f"{key}_pumping_stations.gpkg"

    corrected_gpkg = (
        CORRECTED_NETWORK_BASE
        / _scenario_corrected_dir_name(key)
        / "final_manual_corrected_network.gpkg"
    )

    print(f"\nScenario: {key}")
    print(f"  Pumping GPKG:   {pumping_gpkg}")
    print(f"  Corrected GPKG: {corrected_gpkg}")

    if not pumping_gpkg.exists():
        print("  [SKIP] Pumping-station GeoPackage not found.")
        continue

    if not corrected_gpkg.exists():
        print("  [WARNING] Corrected network GeoPackage not found. Map will only include pumping-station outputs.")

    # Pumping-station layers
    new_ps = _safe_read(pumping_gpkg, "new_ps_final_candidates")
    raw_ps = _safe_read(pumping_gpkg, "new_ps_raw_candidates")
    existing_all = _safe_read(pumping_gpkg, "existing_ps_all_classified")
    wwtp2 = _safe_read(pumping_gpkg, "wwtp2_point")

    existing_wwtp2 = _filter_existing_ps_by_wwtp(existing_all, target="WWTP2")
    existing_wwtp1 = _filter_existing_ps_by_wwtp(existing_all, target="WWTP1")

    if len(existing_all) == 0:
        print("  [WARNING] existing_ps_all_classified missing. Falling back to old existing PS layers.")
        existing_wwtp2 = _safe_read(pumping_gpkg, "existing_ps_now_wwtp2")
        existing_wwtp1 = _safe_read(pumping_gpkg, "existing_ps_wwtp1")

    # Pipe layers from CorrectedNetwork
    if corrected_gpkg.exists():
        pipe_layers = _read_all_final_pipe_layers(corrected_gpkg)
    else:
        pipe_layers = {
            "existing_wwtp1": (gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED), None),
            "existing_wwtp2": (gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED), None),
            "new_wwtp1": (gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED), None),
            "new_wwtp2": (_safe_read(pumping_gpkg, "new_pipes_reference"), "new_pipes_reference"),
            "combined_final": (gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED), None),
        }

    existing_pipes_wwtp1, existing_pipes_wwtp1_layer = pipe_layers["existing_wwtp1"]
    existing_pipes_wwtp2, existing_pipes_wwtp2_layer = pipe_layers["existing_wwtp2"]
    new_pipes_wwtp1, new_pipes_wwtp1_layer = pipe_layers["new_wwtp1"]
    new_pipes_wwtp2, new_pipes_wwtp2_layer = pipe_layers["new_wwtp2"]
    combined_final_pipes, combined_final_layer = pipe_layers["combined_final"]

    print(f"  New proposed PS: {len(new_ps)}")
    print(f"  Existing PS total classified: {len(existing_all)}")
    print(f"  Existing PS now WWTP2: {len(existing_wwtp2)}")
    print(f"  Existing PS WWTP1: {len(existing_wwtp1)}")
    print(f"  Existing pipes WWTP1: {len(existing_pipes_wwtp1)} | layer: {existing_pipes_wwtp1_layer}")
    print(f"  Existing pipes WWTP2: {len(existing_pipes_wwtp2)} | layer: {existing_pipes_wwtp2_layer}")
    print(f"  New pipes WWTP1: {len(new_pipes_wwtp1)} | layer: {new_pipes_wwtp1_layer}")
    print(f"  New pipes WWTP2: {len(new_pipes_wwtp2)} | layer: {new_pipes_wwtp2_layer}")
    print(f"  Combined final pipes fallback: {len(combined_final_pipes)} | layer: {combined_final_layer}")

    center, bounds = _map_center_from_layers(
        [
            existing_pipes_wwtp1,
            existing_pipes_wwtp2,
            new_pipes_wwtp1,
            new_pipes_wwtp2,
            combined_final_pipes,
            new_ps,
            raw_ps,
            existing_wwtp2,
            existing_wwtp1,
            wwtp2,
        ],
        fallback_point=WWTP2_POINT,
    )

    m = folium.Map(
        location=center,
        zoom_start=12,
        tiles="CartoDB positron",
        control_scale=True,
    )

    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Satellite",
        overlay=False,
        control=True,
    ).add_to(m)

    title_html = f"""
    <div style='position: fixed; top: 10px; left: 60px; z-index:9999;
                background-color: white; padding: 10px; border: 2px solid #444;
                font-size: 14px;'>
      <b>Complete Network + Pumping Stations</b><br>
      Scenario: {key}<br>
      Label: {label}<br>
      Family: {family}<br>
      New PS: {len(new_ps)}<br>
      Existing PS now WWTP2: {len(existing_wwtp2)} / {len(existing_all)}
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))

    # --------------------------------------------------------
    # Pipe layers
    # --------------------------------------------------------
    if SHOW_ALL_PIPES:
        _add_line_layer(
            m,
            existing_pipes_wwtp1,
            "Existing pipes WWTP1",
            color=COLOR_EXISTING_PIPES_WWTP1,
            weight=2,
            opacity=0.65,
            show=True,
        )

        _add_line_layer(
            m,
            existing_pipes_wwtp2,
            "Existing pipes WWTP2",
            color=COLOR_EXISTING_PIPES_WWTP2,
            weight=2,
            opacity=0.65,
            show=True,
        )

        _add_line_layer(
            m,
            new_pipes_wwtp1,
            "New pipes WWTP1",
            color=COLOR_NEW_PIPES_WWTP1,
            weight=3,
            opacity=0.85,
            show=True,
        )

        _add_line_layer(
            m,
            new_pipes_wwtp2,
            "New pipes WWTP2",
            color=COLOR_NEW_PIPES_WWTP2,
            weight=3,
            opacity=0.85,
            show=True,
        )

        # Fallback only useful if individual layers were not available.
        if (
            len(existing_pipes_wwtp1) == 0
            and len(existing_pipes_wwtp2) == 0
            and len(new_pipes_wwtp1) == 0
            and len(new_pipes_wwtp2) == 0
            and len(combined_final_pipes) > 0
        ):
            _add_line_layer(
                m,
                combined_final_pipes,
                "Combined final pipes fallback",
                color=COLOR_OTHER_PIPES,
                weight=2,
                opacity=0.70,
                show=True,
            )

    # --------------------------------------------------------
    # Pumping-station layers
    # --------------------------------------------------------
    _add_point_layer_no_cluster(
        m,
        new_ps,
        "NEW proposed pumping stations",
        marker_color=COLOR_NEW_PS,
        fill_color=COLOR_NEW_PS,
        radius=7,
        fields=[
            "station_id",
            "scenario",
            "algorithm_family",
            "reason",
            "screening_reason",
            "depth_excess_m",
            "ground_elev_m",
            "upstream_pipe_length_m",
            "candidate_priority_score",
        ],
        show=True,
    )

    _add_point_layer_no_cluster(
        m,
        existing_wwtp2,
        "EXISTING pumping stations now assigned to WWTP2",
        marker_color=COLOR_EXISTING_PS_NOW_WWTP2,
        fill_color=COLOR_EXISTING_PS_NOW_WWTP2,
        radius=7,
        fields=[
            "nearest_wwtp_area",
            "final_wwtp_assignment",
            "wwtp_assignment",
            "assigned_wwtp",
            "dist_to_wwtp2_pipes_m",
            "dist_to_wwtp1_pipes_m",
            "Name",
            "name",
            "ID",
            "id",
            "ps_source_file",
        ],
        show=True,
    )

    if SHOW_EXISTING_WWTP1:
        _add_point_layer_no_cluster(
            m,
            existing_wwtp1,
            "Existing pumping stations remaining WWTP1",
            marker_color=COLOR_EXISTING_PS_WWTP1,
            fill_color=COLOR_EXISTING_PS_WWTP1,
            radius=5,
            fields=[
                "nearest_wwtp_area",
                "final_wwtp_assignment",
                "wwtp_assignment",
                "assigned_wwtp",
                "dist_to_wwtp2_pipes_m",
                "dist_to_wwtp1_pipes_m",
                "Name",
                "name",
                "ID",
                "id",
                "ps_source_file",
            ],
            show=True,
        )

    if SHOW_RAW_CANDIDATES:
        _add_point_layer_no_cluster(
            m,
            raw_ps,
            "Raw candidate pumping stations",
            marker_color=COLOR_RAW_CANDIDATE,
            fill_color=COLOR_RAW_CANDIDATE,
            radius=4,
            fields=[
                "station_id",
                "reason",
                "depth_excess_m",
                "upstream_pipe_length_m",
                "candidate_priority_score",
            ],
            show=False,
        )

    _add_point_layer_no_cluster(
        m,
        wwtp2,
        "WWTP2 point",
        marker_color=COLOR_WWTP2,
        fill_color=COLOR_WWTP2,
        radius=9,
        fields=[
            "name",
            "WWTP2_LON",
            "WWTP2_LAT",
        ],
        show=True,
    )

    if bounds:
        try:
            m.fit_bounds(bounds)
        except Exception:
            pass

    _add_complete_legend(m, title="Complete Network Legend")
    _add_map_controls(m)

    html_path = MAP_DIR_COMPLETE / f"{key}_COMPLETE_network_pumping_stations_map.html"
    m.save(str(html_path))

    print(f"  HTML complete map saved: {html_path}")

    map_rows.append({
        "scenario": key,
        "label": label,
        "algorithm_family": family,
        "new_ps_final_count": len(new_ps),
        "raw_ps_candidate_count": len(raw_ps),
        "existing_ps_total_classified_count": len(existing_all),
        "existing_ps_now_wwtp2_count": len(existing_wwtp2),
        "existing_ps_wwtp1_count": len(existing_wwtp1),
        "existing_pipes_wwtp1_count": len(existing_pipes_wwtp1),
        "existing_pipes_wwtp2_count": len(existing_pipes_wwtp2),
        "new_pipes_wwtp1_count": len(new_pipes_wwtp1),
        "new_pipes_wwtp2_count": len(new_pipes_wwtp2),
        "combined_final_pipes_count": len(combined_final_pipes),
        "corrected_gpkg": str(corrected_gpkg),
        "pumping_gpkg": str(pumping_gpkg),
        "html_complete_map": str(html_path),
    })

    # For global overview
    for gdf, map_type in [
        (new_ps, "new_ps_final"),
        (existing_wwtp2, "existing_ps_now_wwtp2"),
        (existing_wwtp1, "existing_ps_wwtp1"),
    ]:
        if len(gdf) > 0:
            tmp = gdf.copy()
            tmp["scenario"] = key
            tmp["map_type"] = map_type
            overview_layers.append(tmp)

    for gdf, map_type in [
        (existing_pipes_wwtp1, "existing_pipes_wwtp1"),
        (existing_pipes_wwtp2, "existing_pipes_wwtp2"),
        (new_pipes_wwtp1, "new_pipes_wwtp1"),
        (new_pipes_wwtp2, "new_pipes_wwtp2"),
    ]:
        if len(gdf) > 0:
            tmp = gdf.copy()
            tmp["scenario"] = key
            tmp["map_type"] = map_type
            overview_pipe_layers.append(tmp)


# ------------------------------------------------------------
# 9C.7 Global overview map with all scenarios
# ------------------------------------------------------------
if overview_layers:
    overview_gdf = pd.concat(overview_layers, ignore_index=True)
    overview_gdf = gpd.GeoDataFrame(
        overview_gdf,
        geometry="geometry",
        crs=CRS_PROJECTED,
    )
else:
    overview_gdf = gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED)

if overview_pipe_layers:
    overview_pipes_gdf = pd.concat(overview_pipe_layers, ignore_index=True)
    overview_pipes_gdf = gpd.GeoDataFrame(
        overview_pipes_gdf,
        geometry="geometry",
        crs=CRS_PROJECTED,
    )
else:
    overview_pipes_gdf = gpd.GeoDataFrame(geometry=[], crs=CRS_PROJECTED)

center, bounds = _map_center_from_layers(
    [overview_gdf, overview_pipes_gdf],
    fallback_point=WWTP2_POINT,
)

overview_map = folium.Map(
    location=center,
    zoom_start=11,
    tiles="CartoDB positron",
    control_scale=True,
)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(overview_map)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellite",
    overlay=False,
    control=True,
).add_to(overview_map)

# WWTP2
wwtp2_gdf = gpd.GeoDataFrame(
    [{
        "name": "WWTP2",
        "WWTP2_LON": WWTP2_LON,
        "WWTP2_LAT": WWTP2_LAT,
    }],
    geometry=[WWTP2_POINT],
    crs=CRS_PROJECTED,
)

_add_point_layer_no_cluster(
    overview_map,
    wwtp2_gdf,
    "WWTP2 point",
    marker_color=COLOR_WWTP2,
    fill_color=COLOR_WWTP2,
    radius=9,
    fields=["name", "WWTP2_LON", "WWTP2_LAT"],
    show=True,
)

# Add pipes by scenario and type
for sc in SCENARIOS:
    key = _scenario_key(sc)

    if len(overview_pipes_gdf) == 0:
        continue

    for pipe_type, label, color, weight, show_default in [
        ("existing_pipes_wwtp1", "Existing pipes WWTP1", COLOR_EXISTING_PIPES_WWTP1, 2, False),
        ("existing_pipes_wwtp2", "Existing pipes WWTP2", COLOR_EXISTING_PIPES_WWTP2, 2, False),
        ("new_pipes_wwtp1", "New pipes WWTP1", COLOR_NEW_PIPES_WWTP1, 3, False),
        ("new_pipes_wwtp2", "New pipes WWTP2", COLOR_NEW_PIPES_WWTP2, 3, True),
    ]:
        pipe_s = overview_pipes_gdf[
            (overview_pipes_gdf["scenario"] == key)
            & (overview_pipes_gdf["map_type"] == pipe_type)
        ].copy()

        _add_line_layer(
            overview_map,
            pipe_s,
            f"{key} - {label}",
            color=color,
            weight=weight,
            opacity=0.70,
            show=show_default,
        )

# Add points by scenario
for sc in SCENARIOS:
    key = _scenario_key(sc)

    if len(overview_gdf) == 0:
        continue

    new_s = overview_gdf[
        (overview_gdf["scenario"] == key)
        & (overview_gdf["map_type"] == "new_ps_final")
    ].copy()

    ex2_s = overview_gdf[
        (overview_gdf["scenario"] == key)
        & (overview_gdf["map_type"] == "existing_ps_now_wwtp2")
    ].copy()

    ex1_s = overview_gdf[
        (overview_gdf["scenario"] == key)
        & (overview_gdf["map_type"] == "existing_ps_wwtp1")
    ].copy()

    _add_point_layer_no_cluster(
        overview_map,
        new_s,
        f"{key} - NEW proposed PS",
        marker_color=COLOR_NEW_PS,
        fill_color=COLOR_NEW_PS,
        radius=7,
        fields=[
            "station_id",
            "scenario",
            "algorithm_family",
            "reason",
            "depth_excess_m",
            "upstream_pipe_length_m",
            "candidate_priority_score",
        ],
        show=True,
    )

    _add_point_layer_no_cluster(
        overview_map,
        ex2_s,
        f"{key} - EXISTING PS now assigned to WWTP2",
        marker_color=COLOR_EXISTING_PS_NOW_WWTP2,
        fill_color=COLOR_EXISTING_PS_NOW_WWTP2,
        radius=7,
        fields=[
            "scenario",
            "nearest_wwtp_area",
            "final_wwtp_assignment",
            "dist_to_wwtp2_pipes_m",
            "dist_to_wwtp1_pipes_m",
            "Name",
            "name",
            "ID",
            "id",
        ],
        show=True,
    )

    _add_point_layer_no_cluster(
        overview_map,
        ex1_s,
        f"{key} - Existing PS remaining WWTP1",
        marker_color=COLOR_EXISTING_PS_WWTP1,
        fill_color=COLOR_EXISTING_PS_WWTP1,
        radius=5,
        fields=[
            "scenario",
            "nearest_wwtp_area",
            "final_wwtp_assignment",
            "dist_to_wwtp2_pipes_m",
            "dist_to_wwtp1_pipes_m",
            "Name",
            "name",
            "ID",
            "id",
        ],
        show=False,
    )

if bounds:
    try:
        overview_map.fit_bounds(bounds)
    except Exception:
        pass

_add_complete_legend(overview_map, title="Complete Network Legend - Global Overview")
_add_map_controls(overview_map)

overview_html = MAP_DIR_COMPLETE / "ALL_SCENARIOS_COMPLETE_network_pumping_stations_overview.html"
overview_map.save(str(overview_html))

print(f"\nGlobal complete overview map saved: {overview_html}")

# ------------------------------------------------------------
# 9C.8 Save layer-count summary
# ------------------------------------------------------------
map_summary_df = pd.DataFrame(map_rows)

map_summary_csv = MAP_DIR_COMPLETE / "complete_network_map_layer_counts.csv"
map_summary_df.to_csv(map_summary_csv, index=False)

print("\nComplete map layer-count summary:")
display(map_summary_df)

print(f"\nComplete map summary CSV saved: {map_summary_csv}")
print(f"\nAll complete maps saved in:\n  {MAP_DIR_COMPLETE}")

print("=" * 110)
print("BLOCK 9C COMPLETED")
print("=" * 110)

BLOCK 9C -- COMPLETE INTERACTIVE MAPS WITH ALL PIPES

Scenario: dijkstra
  Pumping GPKG:   C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\dijkstra_pumping_stations.gpkg
  Corrected GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\After_Manual_Networks_Correction\dijkstra_final_manual_correction\final_manual_corrected_network.gpkg
  New proposed PS: 30
  Existing PS total classified: 22
  Existing PS now WWTP2: 13
  Existing PS WWTP1: 9
  Existing pipes WWTP1: 3713 | layer: existing_pipes_WWTP1_final
  Existing pipes WWTP2: 2695 | layer: existing_pipes_WWTP2_final_manual
  New pipes WWTP1: 0 | layer: None
  New pipes WWTP2: 16738 | layer: new_pipes_WWTP2_final
  Combined final pipes fallback: 0 | layer: None
  HTML complete map saved: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\interactive_maps_complete_network\dijkstra_COMPLETE_network_pumping_stations_map.html

Scenario: prim_steiner
  Pumping GPKG:

,scenario,label,algorithm_family,new_ps_final_count,raw_ps_candidate_count,existing_ps_total_classified_count,existing_ps_now_wwtp2_count,existing_ps_wwtp1_count,existing_pipes_wwtp1_count,existing_pipes_wwtp2_count,new_pipes_wwtp1_count,new_pipes_wwtp2_count,combined_final_pipes_count,corrected_gpkg,pumping_gpkg,html_complete_map
0,dijkstra,Rooted Dijkstra,non_budgeted,30,3793,22,13,9,3713,2695,0,16738,0,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...
1,prim_steiner,Prim-Steiner,non_budgeted,29,5634,22,13,9,3551,3003,0,14129,0,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...
2,shared_trunk,Shared-Trunk,budgeted,20,8040,22,13,9,3588,2816,0,9135,0,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...
3,urban_priority,Urban Priority,budgeted,20,2865,22,13,9,3520,3094,0,9342,0,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...
4,municipality_targets,Municipality Targets,budgeted,19,3108,22,13,9,3525,2891,0,9394,0,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...,C:\Users\lucam\OneDrive\Desktop\Cyprus Project...



Complete map summary CSV saved: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\interactive_maps_complete_network\complete_network_map_layer_counts.csv

All complete maps saved in:
  C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\interactive_maps_complete_network
BLOCK 9C COMPLETED


In [11]:
# ============================================================
# BLOCK 10 -- NEW PUMPING-STATION COORDINATES BY SCENARIO
# ============================================================
# Purpose:
#   Extract longitude and latitude of the NEW proposed pumping
#   stations for each scenario.
#
# Input:
#   Results/PunpingStations_Positioning/<scenario>_pumping_stations.gpkg
#   layer: new_ps_final_candidates
#
# Output:
#   Results/PunpingStations_Positioning/chapter6_new_ps_coordinates/
#       new_pumping_stations_coordinates_by_scenario.csv
#       new_pumping_stations_coordinates_by_scenario.xlsx
#       latex_table_new_pumping_stations_coordinates.tex
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path
import fiona

print("=" * 110)
print("BLOCK 10 -- NEW PUMPING-STATION COORDINATES BY SCENARIO")
print("=" * 110)

# ------------------------------------------------------------
# 10.1 Output folder
# ------------------------------------------------------------

NEW_PS_COORD_DIR = OUTPUT_DIR / "chapter6_new_ps_coordinates"
NEW_PS_COORD_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 10.2 Scenario helpers
# ------------------------------------------------------------

def _scenario_key_b10(sc):
    if isinstance(sc, dict):
        return sc.get("key", sc.get("scenario_key", sc.get("scenario", "UNKNOWN")))
    return str(sc)


def _scenario_label_b10(sc):
    if isinstance(sc, dict):
        return sc.get("label", sc.get("display_name", _scenario_key_b10(sc)))
    return str(sc)


def _scenario_family_b10(sc):
    if isinstance(sc, dict):
        return sc.get("algorithm_family", "UNKNOWN")
    return "UNKNOWN"


def _layer_exists_b10(gpkg, layer):
    try:
        return layer in fiona.listlayers(str(gpkg))
    except Exception:
        return False


def _safe_read_b10(gpkg, layer, target_crs=CRS_PROJECTED):
    gpkg = Path(gpkg)

    if not gpkg.exists() or not _layer_exists_b10(gpkg, layer):
        return gpd.GeoDataFrame(geometry=[], crs=target_crs)

    gdf = gpd.read_file(gpkg, layer=layer)

    if gdf.crs is None:
        gdf = gdf.set_crs(target_crs)
    else:
        gdf = gdf.to_crs(target_crs)

    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
    return gdf


def _latex_escape_b10(x):
    if pd.isna(x):
        return ""

    x = str(x)
    repl = {
        "\\": r"\textbackslash{}",
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
        "{": r"\{",
        "}": r"\}",
    }

    for old, new in repl.items():
        x = x.replace(old, new)

    return x


# ------------------------------------------------------------
# 10.3 Extract coordinates
# ------------------------------------------------------------

coord_rows = []

for sc in SCENARIOS:
    key = _scenario_key_b10(sc)
    label = _scenario_label_b10(sc)
    family = _scenario_family_b10(sc)

    pumping_gpkg = OUTPUT_DIR / f"{key}_pumping_stations.gpkg"

    print(f"\nScenario: {key}")
    print(f"  GPKG: {pumping_gpkg}")

    if not pumping_gpkg.exists():
        print("  [SKIP] Pumping-station GeoPackage not found.")
        continue

    new_ps = _safe_read_b10(pumping_gpkg, "new_ps_final_candidates")

    if len(new_ps) == 0:
        print("  No new pumping stations found.")
        continue

    new_ps_proj = new_ps.to_crs(CRS_PROJECTED)
    new_ps_wgs = new_ps.to_crs("EPSG:4326")

    for idx, row in new_ps_wgs.iterrows():
        geom_wgs = row.geometry
        geom_proj = new_ps_proj.loc[idx].geometry

        if geom_wgs is None or geom_wgs.is_empty:
            continue

        if geom_wgs.geom_type != "Point":
            geom_wgs = geom_wgs.centroid

        if geom_proj.geom_type != "Point":
            geom_proj = geom_proj.centroid

        station_id = row.get("station_id", f"PS_NEW_{len(coord_rows)+1:03d}")

        coord_rows.append({
            "scenario_key": key,
            "scenario": label,
            "algorithm_family": family,
            "station_id": station_id,
            "selection_class": row.get("selection_class", ""),
            "is_mandatory_ps": row.get("is_mandatory_ps", ""),
            "mandatory_reason": row.get("mandatory_reason", ""),
            "longitude": geom_wgs.x,
            "latitude": geom_wgs.y,
            "x_projected": geom_proj.x,
            "y_projected": geom_proj.y,
            "ground_elev_m": row.get("ground_elev_m", None),
            "depth_excess_m": row.get("depth_excess_m", None),
            "upstream_pipe_length_m": row.get("upstream_pipe_length_m", None),
            "candidate_priority_score": row.get("candidate_priority_score", None),
        })

    print(f"  Extracted new PS: {len(new_ps)}")

new_ps_coords = pd.DataFrame(coord_rows)

if len(new_ps_coords) == 0:
    print("\nNo new pumping-station coordinates extracted.")
else:
    # Stable order
    scenario_order = [
        "dijkstra",
        "prim_steiner",
        "shared_trunk",
        "urban_priority",
        "municipality_targets",
    ]
    new_ps_coords["_scenario_sort"] = new_ps_coords["scenario_key"].apply(
        lambda x: scenario_order.index(x) if x in scenario_order else 999
    )

    new_ps_coords = (
        new_ps_coords
        .sort_values(["_scenario_sort", "station_id"])
        .drop(columns="_scenario_sort")
        .reset_index(drop=True)
    )

    # Save CSV and Excel
    csv_path = NEW_PS_COORD_DIR / "new_pumping_stations_coordinates_by_scenario.csv"
    xlsx_path = NEW_PS_COORD_DIR / "new_pumping_stations_coordinates_by_scenario.xlsx"

    new_ps_coords.to_csv(csv_path, index=False)
    new_ps_coords.to_excel(xlsx_path, index=False)

    print("\nSaved:")
    print(f"  CSV : {csv_path}")
    print(f"  XLSX: {xlsx_path}")

    print("\nNEW PUMPING-STATION COORDINATES")
    display(new_ps_coords)

    # --------------------------------------------------------
    # 10.4 LaTeX table export
    # --------------------------------------------------------
    # Compact table: scenario, station ID, class, lon, lat.
    # For the thesis, this can go in appendix if too long.

    latex_lines = []
    latex_lines.append(r"\begin{table}[H]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Coordinates of the new proposed pumping stations by scenario.}")
    latex_lines.append(r"\label{tab:new-ps-coordinates}")
    latex_lines.append(r"\scriptsize")
    latex_lines.append(r"\begin{tabular}{llllrr}")
    latex_lines.append(r"\toprule")
    latex_lines.append(r"Scenario & Station ID & Class & Mandatory & Longitude & Latitude \\")
    latex_lines.append(r"\midrule")

    last_scenario = None

    for _, row in new_ps_coords.iterrows():
        scenario = row["scenario"]

        if last_scenario is not None and scenario != last_scenario:
            latex_lines.append(r"\midrule")

        mandatory = row.get("is_mandatory_ps", "")
        if isinstance(mandatory, bool):
            mandatory_txt = "Yes" if mandatory else "No"
        else:
            mandatory_txt = str(mandatory)

        latex_lines.append(
            f"{_latex_escape_b10(scenario)} & "
            f"{_latex_escape_b10(row['station_id'])} & "
            f"{_latex_escape_b10(row.get('selection_class', ''))} & "
            f"{_latex_escape_b10(mandatory_txt)} & "
            f"{float(row['longitude']):.6f} & "
            f"{float(row['latitude']):.6f} \\\\"
        )

        last_scenario = scenario

    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"\end{table}")

    latex_path = NEW_PS_COORD_DIR / "latex_table_new_pumping_stations_coordinates.tex"
    latex_path.write_text("\n".join(latex_lines), encoding="utf-8")

    print(f"\nLaTeX table saved:\n  {latex_path}")

print("\nBLOCK 10 COMPLETED")
print("=" * 110)

BLOCK 10 -- NEW PUMPING-STATION COORDINATES BY SCENARIO

Scenario: dijkstra
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\dijkstra_pumping_stations.gpkg
  Extracted new PS: 30

Scenario: prim_steiner
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\prim_steiner_pumping_stations.gpkg
  Extracted new PS: 29

Scenario: shared_trunk
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\shared_trunk_pumping_stations.gpkg
  Extracted new PS: 20

Scenario: urban_priority
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\urban_priority_pumping_stations.gpkg
  Extracted new PS: 20

Scenario: municipality_targets
  GPKG: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\municipality_targets_pumping_stations.gpkg
  Extracted new PS: 19

Saved:
  CSV : C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Resu

,scenario_key,scenario,algorithm_family,station_id,selection_class,is_mandatory_ps,mandatory_reason,longitude,latitude,x_projected,y_projected,ground_elev_m,depth_excess_m,upstream_pipe_length_m,candidate_priority_score
0,dijkstra,Rooted Dijkstra,non_budgeted,PS_NEW_001,mandatory,True,mandatory_high_depth_excess>= 5.0m;mandatory_e...,33.645963,34.980433,558959.698,3871063.753,108.590424,51.790538,409653.020154,322.650433
1,dijkstra,Rooted Dijkstra,non_budgeted,PS_NEW_002,mandatory,True,mandatory_high_depth_excess>= 5.0m;mandatory_e...,33.640974,35.000218,558490.168,3873254.846,114.833313,53.783959,332641.800255,322.107167
2,dijkstra,Rooted Dijkstra,non_budgeted,PS_NEW_003,mandatory,True,mandatory_high_depth_excess>= 5.0m;mandatory_e...,33.598240,34.989668,554597.575,3872060.736,107.088943,34.146645,466598.204629,222.690952
3,dijkstra,Rooted Dijkstra,non_budgeted,PS_NEW_004,mandatory,True,mandatory_high_depth_excess>= 5.0m;mandatory_e...,33.572014,34.935723,552238.346,3866064.357,60.992189,21.480269,113982.428055,122.381439
4,dijkstra,Rooted Dijkstra,non_budgeted,PS_NEW_005,mandatory,True,mandatory_high_depth_excess>= 5.0m;mandatory_e...,33.605477,34.982423,555262.918,3871261.280,83.106613,16.500852,491538.855114,118.894139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,municipality_targets,Municipality Targets,budgeted,PS_NEW_015,mandatory,True,mandatory_adverse_terrain_rise_with_significan...,33.605506,34.936341,555296.503,3866150.902,29.496201,3.006941,128860.388383,18.126049
114,municipality_targets,Municipality Targets,budgeted,PS_NEW_016,mandatory,True,mandatory_adverse_terrain_rise_with_significan...,33.574162,34.935353,552434.692,3866024.449,62.014023,4.933252,10653.828134,18.066136
115,municipality_targets,Municipality Targets,budgeted,PS_NEW_017,mandatory,True,mandatory_existing_PS_like_terrain_network_sig...,33.691208,34.995231,563078.033,3872732.450,22.477132,3.564572,57404.320609,17.900300
116,municipality_targets,Municipality Targets,budgeted,PS_NEW_018,mandatory,True,mandatory_adverse_terrain_rise_with_significan...,33.637653,34.952413,558220.938,3867951.454,9.735378,1.960037,186677.444285,13.455447



LaTeX table saved:
  C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\chapter6_new_ps_coordinates\latex_table_new_pumping_stations_coordinates.tex

BLOCK 10 COMPLETED


In [12]:
# ============================================================
# BLOCK -- FINAL HTML MAPS FOR PUMPING-STATION SCENARIOS
# ============================================================
# Creates one interactive HTML map per scenario showing:
#   - existing conduits assigned to WWTP1  -> blue
#   - existing conduits assigned to WWTP2  -> orange
#   - new conduits                         -> red
#   - existing pumping stations -> WWTP1   -> purple
#   - existing pumping stations -> WWTP2   -> green
#   - new pumping stations                 -> yellow
#   - WWTP2 location marker                -> blue marker only
#
# Output:
#   Results/PunpingStations_Positioning/final_interactive_maps/
#       <scenario>_pumping_station_final_map.html
# ============================================================

import folium
import geopandas as gpd
import pandas as pd
from pathlib import Path
import fiona

print("=" * 120)
print("FINAL HTML MAPS FOR PUMPING-STATION SCENARIOS")
print("=" * 120)

# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path(r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning")

if "MANUAL_CORRECTION_DIR" not in globals():
    MANUAL_CORRECTION_DIR = Path(
        r"C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\After_Manual_Networks_Correction"
    )

MAP_DIR = OUTPUT_DIR / "final_interactive_maps"
MAP_DIR.mkdir(parents=True, exist_ok=True)

# Real WWTP2 coordinates
WWTP2_LONLAT = (33.611278405275385, 34.978016504185796)
WWTP2_LATLON = (34.978016504185796, 33.611278405275385)

# ------------------------------------------------------------
# 1. COLORS
# ------------------------------------------------------------

COLOR_EXISTING_WWTP1 = "blue"
COLOR_EXISTING_WWTP2 = "orange"
COLOR_NEW_PIPES      = "red"

COLOR_PS_WWTP1 = "purple"
COLOR_PS_WWTP2 = "green"
COLOR_NEW_PS   = "#FFD700"   # yellow / gold

COLOR_BOUNDARY = "black"

# ------------------------------------------------------------
# 2. SCENARIO DEFINITIONS
# ------------------------------------------------------------

SCENARIO_DIR_MAP = {
    "dijkstra": "dijkstra_final_manual_correction",
    "prim_steiner": "prim_steiner_final_manual_correction",
    "shared_trunk": "shared_trunk_final_manual_correction",
    "urban_priority": "urban_priority_final_manual_correction",
    "municipality_targets": "municipality_targets_final_manual_correction",
}

SCENARIO_LABELS = {
    "dijkstra": "Dijkstra",
    "prim_steiner": "Prim-Steiner",
    "shared_trunk": "Shared-Trunk",
    "urban_priority": "Urban Priority",
    "municipality_targets": "Municipality Targets",
}

if "SCENARIOS" in globals():
    scenario_keys = []
    for sc in SCENARIOS:
        if isinstance(sc, dict):
            scenario_keys.append(sc.get("key", sc.get("scenario_key", "")))
        else:
            scenario_keys.append(str(sc))
    scenario_keys = [s for s in scenario_keys if s in SCENARIO_DIR_MAP]
else:
    scenario_keys = list(SCENARIO_DIR_MAP.keys())

# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def _list_layers(gpkg_path):
    try:
        return list(fiona.listlayers(str(gpkg_path)))
    except Exception:
        return []

def _layer_exists(gpkg_path, layer_name):
    return layer_name in _list_layers(gpkg_path)

def _read_layer_if_exists(gpkg_path, layer_name, target_crs=None):
    gpkg_path = Path(gpkg_path)
    if not gpkg_path.exists():
        return gpd.GeoDataFrame(geometry=[], crs=target_crs)

    if not _layer_exists(gpkg_path, layer_name):
        return gpd.GeoDataFrame(geometry=[], crs=target_crs)

    gdf = gpd.read_file(gpkg_path, layer=layer_name)

    if target_crs is not None:
        if gdf.crs is None:
            gdf = gdf.set_crs(target_crs)
        else:
            gdf = gdf.to_crs(target_crs)

    return gdf

def _read_first_existing_layer(gpkg_path, layer_candidates, target_crs=None):
    for lyr in layer_candidates:
        gdf = _read_layer_if_exists(gpkg_path, lyr, target_crs=target_crs)
        if len(gdf) > 0:
            return gdf, lyr
    return gpd.GeoDataFrame(geometry=[], crs=target_crs), None

def _to_wgs84(gdf):
    if gdf is None or len(gdf) == 0:
        return gdf
    if gdf.crs is None:
        raise RuntimeError("A GeoDataFrame has no CRS; cannot convert to EPSG:4326.")
    return gdf.to_crs(epsg=4326)

def _style_line(color, weight=3, opacity=0.95):
    return lambda feature: {
        "color": color,
        "weight": weight,
        "opacity": opacity
    }

def _get_bounds_from_gdfs(gdfs):
    minx_list, miny_list, maxx_list, maxy_list = [], [], [], []

    for gdf in gdfs:
        if gdf is not None and len(gdf) > 0:
            bounds = gdf.total_bounds
            minx_list.append(bounds[0])
            miny_list.append(bounds[1])
            maxx_list.append(bounds[2])
            maxy_list.append(bounds[3])

    if len(minx_list) == 0:
        return None

    return [[min(miny_list), min(minx_list)], [max(maxy_list), max(maxx_list)]]

def _normalize_text(s):
    return (
        s.astype(str)
        .str.upper()
        .str.replace("_", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

def _split_existing_ps_from_classified_layer(existing_all):
    """
    Split existing pumping stations into WWTP1 / WWTP2 using available assignment columns.
    """
    if existing_all is None or len(existing_all) == 0:
        return (
            gpd.GeoDataFrame(geometry=[], crs=existing_all.crs if existing_all is not None else None),
            gpd.GeoDataFrame(geometry=[], crs=existing_all.crs if existing_all is not None else None)
        )

    g = existing_all.copy()

    assignment_cols = [
        "final_wwtp_assignment",
        "nearest_wwtp_area",
        "wwtp_assignment",
        "assigned_wwtp",
        "service_area",
    ]

    for col in assignment_cols:
        if col in g.columns:
            vals = _normalize_text(g[col])

            mask_wwtp2 = vals.str.contains("WWTP2", na=False) | vals.str.contains("NORTH", na=False)
            mask_wwtp1 = vals.str.contains("WWTP1", na=False) | vals.str.contains("SOUTH", na=False)

            return g[mask_wwtp1].copy(), g[mask_wwtp2].copy()

    return (
        gpd.GeoDataFrame(geometry=[], crs=g.crs),
        gpd.GeoDataFrame(geometry=[], crs=g.crs)
    )

# ------------------------------------------------------------
# 4. CREATE MAPS
# ------------------------------------------------------------

for scenario_key in scenario_keys:
    print("\n" + "-" * 120)
    print(f"SCENARIO: {scenario_key}")
    print("-" * 120)

    corrected_dir = MANUAL_CORRECTION_DIR / SCENARIO_DIR_MAP[scenario_key]
    corrected_gpkg = corrected_dir / "final_manual_corrected_network.gpkg"
    pumping_gpkg   = OUTPUT_DIR / f"{scenario_key}_pumping_stations.gpkg"

    print(f"Corrected network GPKG : {corrected_gpkg}")
    print(f"Pumping stations GPKG  : {pumping_gpkg}")

    if not corrected_gpkg.exists():
        print(f"[SKIP] Corrected network GPKG not found for {scenario_key}")
        continue

    if not pumping_gpkg.exists():
        print(f"[SKIP] Pumping stations GPKG not found for {scenario_key}")
        continue

    # --------------------------------------------------------
    # 4.1 Read pipe layers
    # --------------------------------------------------------

    existing_wwtp1, lyr_existing_wwtp1 = _read_first_existing_layer(
        corrected_gpkg,
        [
            "existing_conduits_WWTP1_final",
            "existing_pipes_WWTP1_final",
            "existing_pipes_WWTP1_final_manual",
        ]
    )

    existing_wwtp2, lyr_existing_wwtp2 = _read_first_existing_layer(
        corrected_gpkg,
        [
            "existing_conduits_WWTP2_final",
            "existing_pipes_WWTP2_final",
            "existing_pipes_WWTP2_final_manual",
        ]
    )

    new_pipes, lyr_new_pipes = _read_first_existing_layer(
        corrected_gpkg,
        [
            "new_pipes_WWTP2_final",
            "new_pipes_WWTP2_final_manual",
            "new_pipes_reference",
        ]
    )

    north_boundaries, lyr_north = _read_first_existing_layer(
        corrected_gpkg,
        [
            "north_municipalities_reference",
            "northern_municipalities",
            "municipal_boundaries",
        ]
    )

    # --------------------------------------------------------
    # 4.2 Read pumping-station layers
    # --------------------------------------------------------

    # Existing PS already classified in pumping notebook
    existing_all, lyr_existing_all = _read_first_existing_layer(
        pumping_gpkg,
        [
            "existing_ps_all_classified",
        ]
    )

    if len(existing_all) > 0:
        ps_wwtp1, ps_wwtp2 = _split_existing_ps_from_classified_layer(existing_all)
    else:
        ps_wwtp1, _ = _read_first_existing_layer(
            pumping_gpkg,
            [
                "existing_ps_wwtp1",
                "existing_pumping_stations_WWTP1_final",
            ]
        )
        ps_wwtp2, _ = _read_first_existing_layer(
            pumping_gpkg,
            [
                "existing_ps_now_wwtp2",
                "existing_pumping_stations_WWTP2_final",
            ]
        )

    # New proposed pumping stations
    new_ps, lyr_new_ps = _read_first_existing_layer(
        pumping_gpkg,
        [
            "new_ps_final_candidates",
            "new_pumping_stations",
            "new_ps",
        ]
    )

    # --------------------------------------------------------
    # 4.3 Convert to WGS84
    # --------------------------------------------------------

    if len(existing_wwtp1) > 0:
        existing_wwtp1 = _to_wgs84(existing_wwtp1)

    if len(existing_wwtp2) > 0:
        existing_wwtp2 = _to_wgs84(existing_wwtp2)

    if len(new_pipes) > 0:
        new_pipes = _to_wgs84(new_pipes)

    if len(north_boundaries) > 0:
        north_boundaries = _to_wgs84(north_boundaries)

    if len(ps_wwtp1) > 0:
        ps_wwtp1 = _to_wgs84(ps_wwtp1)

    if len(ps_wwtp2) > 0:
        ps_wwtp2 = _to_wgs84(ps_wwtp2)

    if len(new_ps) > 0:
        new_ps = _to_wgs84(new_ps)

    # --------------------------------------------------------
    # 4.4 Create base map
    # --------------------------------------------------------

    center_lat, center_lon = WWTP2_LATLON[0], WWTP2_LATLON[1]

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles="OpenStreetMap",
        control_scale=True
    )

    # --------------------------------------------------------
    # 4.5 Municipal boundaries
    # --------------------------------------------------------

    if len(north_boundaries) > 0:
        folium.GeoJson(
            north_boundaries,
            name="North municipalities",
            style_function=lambda feature: {
                "color": COLOR_BOUNDARY,
                "weight": 1.1,
                "opacity": 0.8,
                "fillOpacity": 0.0
            },
            tooltip=None
        ).add_to(m)

    # --------------------------------------------------------
    # 4.6 Pipe layers
    # --------------------------------------------------------

    if len(existing_wwtp1) > 0:
        folium.GeoJson(
            existing_wwtp1,
            name="Existing conduit → WWTP1 final",
            style_function=_style_line(COLOR_EXISTING_WWTP1, weight=3, opacity=0.95),
            tooltip=None
        ).add_to(m)

    if len(existing_wwtp2) > 0:
        folium.GeoJson(
            existing_wwtp2,
            name="Existing conduit → WWTP2 final",
            style_function=_style_line(COLOR_EXISTING_WWTP2, weight=3, opacity=0.95),
            tooltip=None
        ).add_to(m)

    if len(new_pipes) > 0:
        folium.GeoJson(
            new_pipes,
            name="New conduit",
            style_function=_style_line(COLOR_NEW_PIPES, weight=3, opacity=0.95),
            tooltip=None
        ).add_to(m)

    # --------------------------------------------------------
    # 4.7 Existing PS -> WWTP1
    # --------------------------------------------------------

    if len(ps_wwtp1) > 0:
        for _, row in ps_wwtp1.iterrows():
            geom = row.geometry
            if geom is not None and not geom.is_empty:
                if geom.geom_type != "Point":
                    geom = geom.centroid

                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=5,
                    color=COLOR_PS_WWTP1,
                    weight=1.5,
                    fill=True,
                    fill_color=COLOR_PS_WWTP1,
                    fill_opacity=0.95,
                    tooltip=None
                ).add_to(m)

    # --------------------------------------------------------
    # 4.8 Existing PS -> WWTP2
    # --------------------------------------------------------

    if len(ps_wwtp2) > 0:
        for _, row in ps_wwtp2.iterrows():
            geom = row.geometry
            if geom is not None and not geom.is_empty:
                if geom.geom_type != "Point":
                    geom = geom.centroid

                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=5,
                    color=COLOR_PS_WWTP2,
                    weight=1.5,
                    fill=True,
                    fill_color=COLOR_PS_WWTP2,
                    fill_opacity=0.95,
                    tooltip=None
                ).add_to(m)

    # --------------------------------------------------------
    # 4.9 New pumping stations
    # --------------------------------------------------------

    if len(new_ps) > 0:
        for _, row in new_ps.iterrows():
            geom = row.geometry
            if geom is not None and not geom.is_empty:
                if geom.geom_type != "Point":
                    geom = geom.centroid

                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=6,
                    color="#B8860B",      # darker outline
                    weight=1.5,
                    fill=True,
                    fill_color=COLOR_NEW_PS,
                    fill_opacity=0.98,
                    tooltip=None
                ).add_to(m)

    # --------------------------------------------------------
    # 4.10 WWTP2 marker (icon only)
    # --------------------------------------------------------

    folium.Marker(
        location=[WWTP2_LATLON[0], WWTP2_LATLON[1]],
        icon=folium.Icon(color="blue", icon="map-marker", prefix="fa"),
        tooltip=None
    ).add_to(m)

    # --------------------------------------------------------
    # 4.11 Legend
    # --------------------------------------------------------

    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        width: 305px;
        z-index: 9999;
        font-size: 13px;
        background-color: white;
        border: 2px solid #888;
        border-radius: 6px;
        padding: 10px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.25);
    ">
    <b>Legend</b><br><br>

    <span style="display:inline-block; width:26px; height:3px; background:{COLOR_EXISTING_WWTP1}; vertical-align:middle;"></span>
    &nbsp;Existing conduit → WWTP1 final<br>

    <span style="display:inline-block; width:26px; height:3px; background:{COLOR_EXISTING_WWTP2}; vertical-align:middle;"></span>
    &nbsp;Existing conduit → WWTP2 final<br>

    <span style="display:inline-block; width:26px; height:3px; background:{COLOR_NEW_PIPES}; vertical-align:middle;"></span>
    &nbsp;New conduit<br>

    <span style="color:{COLOR_PS_WWTP1}; font-size:18px;">&#8226;</span>
    &nbsp;Pumping station → WWTP1<br>

    <span style="color:{COLOR_PS_WWTP2}; font-size:18px;">&#8226;</span>
    &nbsp;Pumping station → WWTP2<br>

    <span style="color:{COLOR_NEW_PS}; font-size:18px;">&#8226;</span>
    &nbsp;New pumping station<br>

    <span style="color:blue; font-size:18px;">&#9873;</span>
    &nbsp;WWTP2 location
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # --------------------------------------------------------
    # 4.12 Fit bounds
    # --------------------------------------------------------

    bounds = _get_bounds_from_gdfs([
        existing_wwtp1,
        existing_wwtp2,
        new_pipes,
        ps_wwtp1,
        ps_wwtp2,
        new_ps,
        north_boundaries
    ])

    if bounds is not None:
        min_lat = min(bounds[0][0], WWTP2_LATLON[0])
        min_lon = min(bounds[0][1], WWTP2_LATLON[1])
        max_lat = max(bounds[1][0], WWTP2_LATLON[0])
        max_lon = max(bounds[1][1], WWTP2_LATLON[1])
        m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    # --------------------------------------------------------
    # 4.13 Save map
    # --------------------------------------------------------

    html_path = MAP_DIR / f"{scenario_key}_pumping_station_final_map.html"
    m.save(str(html_path))

    print(f"[OK] Saved: {html_path}")

print("\nAll final pumping-station maps created successfully.")
print(f"Output folder:\n{MAP_DIR}")
print("=" * 120)

FINAL HTML MAPS FOR PUMPING-STATION SCENARIOS

------------------------------------------------------------------------------------------------------------------------
SCENARIO: dijkstra
------------------------------------------------------------------------------------------------------------------------
Corrected network GPKG : C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\After_Manual_Networks_Correction\dijkstra_final_manual_correction\final_manual_corrected_network.gpkg
Pumping stations GPKG  : C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\dijkstra_pumping_stations.gpkg
[OK] Saved: C:\Users\lucam\OneDrive\Desktop\Cyprus Project\Results\PunpingStations_Positioning\final_interactive_maps\dijkstra_pumping_station_final_map.html

------------------------------------------------------------------------------------------------------------------------
SCENARIO: prim_steiner
--------------------------------------------------------------------